# LASSI-Verify: Emperical Verification

**Argo Gateway API** Jupyter Notebook version

+ ***Post LASSI-Judge version with data collection for pass@k and energy_reduction@k***
+ No translation
+ Source code scripts from:
https://github.com/zjin-lcf/HeCBench/tree/master/src

NVIDIA GPU power measurements: https://pypi.org/project/nvidia-ml-py/  (pynvml)

Added support for AMD GPU device measurements and HIP code (ROCm)

In [1]:
from lassi.gpu_detector import detect_gpu_vendors

# Detect available GPUs
vendors = detect_gpu_vendors()
if not vendors:
    print("❌ No GPUs detected! LASSI requires GPU hardware.")
    raise SystemExit("Pipeline stopped - no GPU hardware found")

detected_vendor = vendors[0]['vendor']
gpu_count = vendors[0]['count']
print(f"🔍 Detected: {detected_vendor.upper()} with {gpu_count} GPU(s)")

# Vendor-specific initialization
if detected_vendor == "nvidia":
    # NVIDIA-specific setup
    import pynvml
    pynvml.nvmlInit()
    print("NVIDIA Driver Version:", pynvml.nvmlSystemGetDriverVersion())
    pynvml.nvmlShutdown()
    
    # Show NVIDIA system info
    !nvidia-smi
    !nvcc --version
    
elif detected_vendor == "amd":
    # AMD-specific setup
    from lassi.amd_power import amd_report_gpu_memory
    
    # Show AMD system info
    !rocminfo | head -20
    !rocm-smi
    !hipcc --version
    
print(f"\n✅ {detected_vendor.upper()} GPU environment ready for LASSI!")

🔍 Detected: NVIDIA with 1 GPU(s)
NVIDIA Driver Version: 570.211.01
Tue Mar 17 09:55:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 5000 Ada Gene...    Off |   00000000:01:00.0 Off |                  Off |
| N/A   39C    P8              3W /   80W |      14MiB /  16376MiB |      0%      Default |
|                                         |                        |     

## Setup

In [2]:
import os
import json
import csv
import time
import shutil
from datetime import datetime
from zoneinfo import ZoneInfo

# Import LASSI modules
import lassi.llm_config
# import lassi.pipeline # The Pipeline script calls the other LASSI libraries not imported here.
import lassi.pipeline_v4
from lassi.format_colors import color
from lassi.prompt_dictionary import PromptDictionary
from lassi.monitor import start_monitor, log_figure

# Enable the autoreload extension
%load_ext autoreload
%reload_ext autoreload

# Reload all modules imported with %aimport every time before executing subsequent code.
%autoreload 1

# We need to reload the libraries while under development
%aimport lassi.format_colors
%aimport lassi.prompt_dictionary
%aimport lassi.llm_config
%aimport lassi.llm_methods
%aimport lassi.pipeline_v4
%aimport lassi.get_power
%aimport lassi.argostream_curl
%aimport lassi.code_manager
%aimport lassi.pipeline_methods
%aimport lassi.verify
%aimport lassi.monitor

In [3]:
# ── LASSI Monitor — live output dashboard ──────────────────────────────────────
# Starts a local web server and streams all pipeline print() output to a browser
# tab in real-time.  Run this cell once after the imports cell above.
# Then open the link below in a new browser tab and keep it visible.
#
# If matplotlib figures are produced (EE task), they appear automatically in a
# sidebar panel on the right side of the dashboard — no extra calls needed.
#
# The dashboard accumulates the full session history, so a late-opened tab
# will still see everything printed since start_monitor() was called.
#
# SSH remote access: forward port 5007 to your local machine, then open
#   http://localhost:5007  in your local browser.
start_monitor()   # default port 5007 — pass port=XXXX to change

'http://localhost:5007'

## Code Generation Experiment - Setup + Run

_____
***STEP 1/2: Input experimental configuration parameters in the following method.***

In [4]:
def experimental_setup():
    # Load config file with source application names and input parameters
    with open("source_apps.json", "r") as f:
        source_apps = json.load(f)
    
    """
    [1] ENTER APPLICATION FOLDER NAME FROM HeCBench
        >> Execution input parameters provided.
    
    *** USE FOR LASSI-Verfify experiment: ***
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
     (this one may show a lot of failures)

    HeCBench apps:
    -- matrix-rotate ["30000", "100"]  ["10000", "1"]  # Usage: %s <matrix size> <repeat>
    -- jacobi [""] (none required)
    -- layout ["2500"] ["1"]
    -- atomicCost ["16", "10"] ["1"]
    -- segment-reduce ["16384", "100"] (a replacement for atomicCost, same "Data compression and reduction" category in HeCBench)
       --> (may be trivial for LASSI-Verify)
    -- dense-embedding ["100000", "128", "100"] ["10000", "8", "1"]
    -- pathfinder ["100000", "10000", "2500"] ["100000", "1000", "5"] ["10000", "1000", "1000"]  # Usage: <column length> <row length> <pyramid_height>
    -- bsearch ["8388608", "1000"] ["10000", "1"]  # Usage: <number of elements> <repeat>
    -- entropy ["10000", "1024", "1"] ["8192", "8192", "500"]   # Usage: <width> <height> <repeat>
    -- colorwheel ["10000", "256", "500000"] ["10000", "8", "1"]  # Usage: <range> <size> <repeat>
    -- randomAccess ["10"] ["1"]  # Usage: <repeat>
    -- lid-driven-cavity [""]
    -- murmurhash3 ["750000, 500"]
    -- chacha20 ["300000"]
    -- jaccard ["1024, 512, 1000"]
    -- keogh ["256", "22500000", "100"]
    -- floydwarshall ["2048", "100", "16"]
    -- extrema ["750"]
    -- threadfence ["100", "250000000"]
    -- all-pairs-distance ["10000"]
    -- marchingCubes ["100"]

    Altis apps:
    -- level1/bfs ["--size 3"]
    -- level2/raytracing [""]
    -- level2/mandelbrot ["--imageSize 1024 --iterations 100000"]

    Miniapps:
    -- miniMDock ["-lfile", "./input/7cpa/7cpa_ligand.pdbqt", "-nrun 500"]
    -- XSBench ["-m", "event", "-l", "1700000000"]
    
    """

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # Enter source application to process through LASSI
    source_collection = "HeCBench"  # "Miniapps" "HeCBench"
    source_app = "jaccard"  # "miniMDock" jacobi
    make_script = False  # Used only for Miniapps
    build_script = False  # False (for HeCBench) Use for Altis only
    

    """
    Valid source code folder names (under source_codes):
    1. HeCBench
    2. altis/src/cuda
    3. miniMDock/bin/
    """
    source_folder = source_apps[source_collection]["source_folder"]  # "altis/src/cuda/level2"    #  "HeCBench" "altis/src/cuda/level1"
    app_folder_name = source_apps[source_collection][source_app]["app_folder"]
    lines_of_code = source_apps[source_collection][source_app]["code_lines"]
    if make_script:
        app_name = source_app  # app_folder_name
        file_name = "Main"  # kernels
    else:
        app_name = "mt_" + app_folder_name + "-cuda_main"
        file_name = ""

    
    """
    [2] SELECT THE CODE TRANSLATION MAPPING DIRECTION EXPERIMENT CONFIGURATION
    
    CUDA -- OMP -- HIP -- SYCL -- FORTRAN
    """   
    # These should be the same if *not* doing translation.
    source_lang = "CUDA"
    target_lang = "CUDA"

    verifyprompt = True  # True or False -- Print out LLM prompt in stdout to preview
    
    # TOGGLE USE CODE KNOWLEDGE CONTEXT FOR TARGET
    use_context_summaries_only = True  # True or False
    few_shot_learning = False  # True or False
    
    # LLM judge code analysis (pre-cursor to LASSI-Verify)
    run_code_analysis = False  # True or False
    
    # THESE WILL NO LONGER BE OPTIONS
    # test_code_first = True  # True or False
    use_context = False  # True or False -- NOT USED IN v2, v3
    use_self_prompting = False  # True or False -- NOT USED IN v2, v3
    baseline_oneshot_goal = True  # True or False -- NOT USED IN v2, v3
    
    # -----------------------------------------------------------------------
    # TASK MODE CONFIGURATION
    # Set these two flags together depending on the task:
    #
    #   ENERGY EFFICIENCY (EE) task:
    #     power_measure    = True   -- measure GPU power/energy during execution
    #     yes_power_feedback = True -- run LASSI self-improvement loop using energy as the optimization metric
    #
    #   RUNTIME PERFORMANCE task:
    #     power_measure    = False  -- skip power measurement, use wall-clock duration instead
    #     yes_power_feedback = True -- run LASSI self-improvement loop using runtime as the optimization metric
    #
    #   CODE TRANSLATION task (one-shot, no iterative improvement):
    #     power_measure    = False  -- no measurement needed
    #     yes_power_feedback = False -- skip feedback loop entirely (generate once, compile, execute, judge, done)
    #     NOTE: translation mode still has a known infinite-loop-on-success issue when yes_power_feedback=False
    # -----------------------------------------------------------------------
    power_measure = False  # True (EE task) | False (runtime or translation task)
    vary_llm_temp = True  # True or False

    # TOGGLE LASSI SELF-IMPROVEMENT FEEDBACK LOOP
    yes_power_feedback = True  # True (EE or runtime task) | False (translation task, one-shot only)

    # -----------------------------------------------------------------------
    # LASSI-VERIFY TIER 2 CONFIGURATION
    # run_tier2=True: insert passthrough observation blocks and validate empirical equivalence
    # tier2_tolerance: relative floating-point tolerance for passthrough comparison (default 1e-6)
    # -----------------------------------------------------------------------
    run_tier2 = True         # True to enable Tier 2 empirical equivalence checking
    tier2_tolerance = 1e-6    # Relative tolerance for passthrough block comparison

    # LASSI-VERIFY TIER 3 CONFIGURATION
    # run_tier3=True: generate metamorphic contract and run variant filter (requires run_tier2=True)
    # tier3_tolerance: relative tolerance for metamorphic relation checks (default 1e-5, slightly looser than Tier 2)
    # -----------------------------------------------------------------------
    run_tier3 = True        # True to enable Tier 3 metamorphic testing
    tier3_tolerance = 1e-5   # Relative tolerance for metamorphic relation comparison

    # -----------------------------------------------------------------------
    # LLM-AS-A-JUDGE CONFIGURATION
    # run_llm_judge=True: enable LLM functional equivalence judge (pre-Tier-1/2 stop-gap)
    # run_llm_judge=False: pure three-tier mode — Tier 1 + Tier 2 determine validity only
    # -----------------------------------------------------------------------
    run_llm_judge = True     # True to enable LLM judge; False for pure three-tier LASSI-Verify mode

    # ASSIGN CODE COMPILE COMMAND
    code_build_script = ""
    code_compiler = ""
    code_compiler_kwds = ""
    if source_lang == "CUDA":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=CUDA DEVICE=GPU CARD=NVIDIA NUMWI=64"
            elif source_app == "XSBench":
                code_compiler = "cuda/make"
        else:
            code_compiler = "nvcc"
            code_compiler_kwds = "-std=c++14 -Xcompiler -Wall -arch=sm_89 -O3"
            # -arch=sm_70 for aiops01 GPU (Argonne)  # -arch=sm_80 for A100s
            # -arch=sm_60 for P100
            # -arch=sm_89 for RTX 5000
        if yes_power_feedback:
            context_knowledge_selection = "cuda_optimization_summary"
            context_fewshot_examples = "cuda_c_few_shot_examples"
        else:
            context_knowledge_selection = "cuda"
    elif source_lang == "OMP":
        # For A100 on Linux_x86_64 or Linux_aarch64
        # code_compiler = "g++" # "/opt/nvidia/hpc_sdk/Linux_aarch64/2024/compilers/bin/nvc++"  # 
        # code_compiler_kwds = "-fopenmp -O3" # "-Wall -O3 -Minfo -mp=gpu -gpu=cc80"  # -c
        # !! TODO - REVIEW: code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc80"  # "-fopenmp -O3 -c"
        # For RTX 5000 Ada on Linux x86_64:
        code_compiler = "/opt/nvidia/hpc_sdk/Linux_x86_64/25.7/compilers/bin/nvc++" # "g++"
        code_compiler_kwds = "-Wall -O3 -Minfo=accel -mp=gpu -gpu=cc89"  # "-fopenmp -O3"
        context_knowledge_selection = "omp"
    elif source_lang == "HIP":
        if build_script:
            code_build_script = "build_" + app_name
        elif make_script:
            if source_app == "miniMDock":
                code_compiler = "make"
                code_compiler_kwds = "API=HIP DEVICE=GPU CARD=AMD NUMWI=64 HIP_PATH=/opt/rocm-6.3.3"
            elif source_app == "XSBench":
                code_compiler = "hip/make"
            else:
                code_compiler_kwds = ""
        else:
            code_compiler = "hipcc"
            code_compiler_kwds = "-std=c++14 -Wall -O3 --amdgpu-target=gfx908"
        if yes_power_feedback:
            context_knowledge_selection = "hip_optimization_summary"
            context_fewshot_examples = "hip_c_few_shot_examples"
        else:
            context_knowledge_selection = "hip"  # TODO

    # TOGGLE EXECUTE PIPELINE
    yes_execute = True  # True / False
    if yes_execute:
        # Enter execution input parameters, if needed
        # A list of parameter string values, e.g., ["10000", "1
        # ****************************
        # ---> EXPERIMENT UPDATE FIELD - Now, automated:
        exe_input_parameters = source_apps[source_collection][source_app]["app_params"]
    else:
        exe_input_parameters = [""]

    
    """
    [3] SELECT THE LLM TO USE FOR THE CODE TRANSLATION EXPERIMENT
    """
    # SELECT A MODEL

    # ****************************
    # ---> EXPERIMENT UPDATE FIELD
    
    # -- FROM ARGO:
    # model = "gpt35"  # (max token = 4096)
    # model = "gpt35large"  # (max token = 16384)
    # model = "gpt4large"  # (max token = 32768)
    # model = "gpt4turbo"  # (max token = 128000)
    # model = "gpto4mini"  # (max token = 128000)
    # model_coder = "gpto4mini"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model_coder = "gpt4o"  # (max token = 128000)
    # model_coder = "gpt4olatest"  # (max token = 128000)
    # model = "gpt4olatest"  # (max token = 200000)
    # model_coder = "gpt4olatest"  # (max token = 200000)
    model = "gpt54"
    model_coder = "gpt54"
    model_judge = "gpt41"  # (max token = 200000)

    # -- FROM CEREBRAS
    # model = "gpt-oss-120b"
    # model_judge = "gpt-oss-120b"
    # model_coder = "qwen-3-coder-480b"
    
    # -- FROM OLLAMA
    # model = "devstral:24b" # (max token = 128000)
    # model_coder = "devstral:24b" # (max token = 128000)
    # model_judge = "devstral:24b" # (max token = 128000)
    # model = "qwq:latest"  # (max token = 128000)
    # model_coder = "qwen2.5-coder:32b"  # (max token = 32768)
    # model_judge = "qwq:latest"  # (max token = 128000)
    # model = "qwq:latest"  # (max token = 131072)
    # model = "gemma3:27b"  # crashing on code gen
    # model_coder = "gemma3:27b"
    # model = "granite3.2:8b-instruct-q8_0"
    # model_coder = "granite-code:34b-instruct-q8_0"
    # model = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-r1:70b"  # (max token = 128000)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model_coder = "deepseek-coder-v2:16b"  # (max token = 131072)
    # model = "dolphin3"  # (max token = 131072)
    # model = "llama3.3:latest"  # (max token = 131072)
    # model = "phi4" # (max token = 16384)
    # model_coder = "codestral:22b"  # (max token = 32768)
    # model = "codestral:22b-v0.1-q8_0"  # (max token = 32768)
    # model = "wizardcoder:33b-v1.1-q8_0"  # (max token = 16384)
    # model_coder = "deepseek-coder:33b-instruct-q8_0"  # (max token = 16384)
    # model = "deepseek-coder-v2:16b-lite-instruct-fp16"  # (max token = 128000)
    # model = "deepseek-coder-v2:16b-lite-instruct-q8_0"
    # model = "phi3:14b"  # (max token = 131072)
    # model_judge = "llama3.3:70b"  # (max token = 128000)
    # model = "qwen3:latest"
    # model_coder = "qwen3:latest"
    # model_judge = "qwen3:latest"  # (max token = 40960)
    # model_coder = "qwen3-coder:30b"  # context 262144
    # model = "gpt-oss:20b"
    # model_judge = "gpt-oss:20b"
  
    # Set maximum context token count size for selected LLM:
    num_ctx = 200000
    num_ctx_coder = 200000

    # Needed only for Argo endpoints
    uri = "local"  # local  remote
    lassi.llm_config.argo_username = "mdearing"  # Argo Gateway username

    # ASSIGN LLM API SOURCE:
    if model in ["gpt54", "gpt5", "gpt35", "gpt35large", "gpt4", "gpt4large", "gpt4turbo", "gpt4o", "gpt4olatest", "gpt41",
                 "gpto1preview", "gpto1mini", "gpto3mini", "gpto4mini", "claudesonnet4", "claudesonnet46"]:
        llm_source = "argo"
    elif model in ["gpt-oss-120b", "qwen-3-coder-480b"]:
        llm_source = "cerebras"
    elif ("GGUF" in model and "hf.co" not in model):
        llm_source = "lmstudio"
    else:
        llm_source = "ollama"

    """
    [4] SET ADDITIONAL PIPELINE PARAMETERS
    """
    experiment = "TEST_FULLEXP_verify_0314-2026_" + model_coder + "_" + app_folder_name + "_runtm_" + detected_vendor.upper()  # Assign any label for experiment name.
    display_response = True
    test_with_error = False
    generate_test_case = False

    # Unload any previously loaded LLMs in the Ollama server 
    # (clear out the memory)
    if llm_source == "ollama":
        lassi.llm_config.unload_ollama_model(model)
        lassi.llm_config.unload_ollama_model(model_coder)

    """
    [5] SET DATA COLLECTION CONFIGURATION
    """
    # Data spreadsheet column headers
    DATAFIELDNAMES = [
        "trial_number",
        "timestamp",
        "experiment",
        "gpu_vendor",
        "app_name",
        "code_line_count",
        "llm",
        "llm_coder",
        "llm_judge",
        "source_v0_energy_J",
        "source_v0_avg_power_W",
        "source_v0_runtime_s",
        "source_verify_runtime_s",  # ★ instrumented source runtime
        "base_v1_energy_J",
        "base_v1_avg_power_W",
        "base_v1_runtime_s",
        "base_v1_valid",
        "base_v1_energy_diff",
        "base_v1_avg_power_diff",
        "base_v1_runtime_diff",
        "zeroshot_runtime_s",        # ★ schema-aligned alias for base_v1_runtime_s
        "zeroshot_compile_status",   # ★ "success" | "compile_error" | "skipped"
        "zeroshot_execute_status",   # ★ "success" | "execute_error" | "skipped"
        # "base_v1_optimization_methods",
        # "base_v2_energy_J",
        # "base_v2_avg_power_W",
        # "base_v2_runtime_s",
        # "base_v2_valid",
        # "base_v2_selfcorrections",
        # "base_v2_energy_diff",
        # "base_v2_avg_power_diff",
        # "base_v2_runtime_diff",
        # "base_v2_optimization_methods",
        "lassiee_energy_J",
        "lassiee_avg_power_W",
        "lassiee_runtime_s",
        "lassiee_valid",
        "lassiee_version",
        "lassiee_llm_temp",
        "lassiee_selfcorrections",
        "lassiee_eeiterations",
        "lassiee_energy_diff",
        "lassiee_avg_power_diff",
        "lassiee_runtime_diff",
        "lassiee_optimization_methods",
        "total_selfcorrections",
        "compile_selfcorrections",   # ★
        "execute_selfcorrections",   # ★
        "tier1_selfcorrections",     # ★
        "tier2_selfcorrections",     # ★
        "tier3_selfcorrections",     # ★
        "global_counter_at_end",     # ★ global verify counter at trial end
        "total_code_versions",       # ★ distinct LASSI code block indices
        "total_code_attempts",       # ★ rows in attempt log for this trial
        "total_eeiterations",
        "pipeline_runtime_min",
        "tier2_pass_count",
        "tier2_fail_total",
        "tier3_pass_count",
        "tier3_fail_total",
        "tier3_active_relations",
        "tier3_inactive_relations",  # ★
        "judge_false_positives",
        "judge_false_negatives",
        "lassiee_clean_status",
        "lassiee_clean_runtime_s",
        "lassiee_clean_runtime_diff",
        "lassiee_clean_energy_J",
        "lassiee_clean_energy_diff",
        "base_llm_tier1_verdict",
        "base_llm_tier2_verdict",
        "base_llm_tier3_verdict",
        "base_llm_verified_valid"
    ]
    
    return (app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder, llm_source, use_context, use_context_summaries_only,
            few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler, code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters,
            experiment, display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal, use_self_prompting, vary_llm_temp,
            verifyprompt, DATAFIELDNAMES, detected_vendor.upper(),
            run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
            run_llm_judge)

# EXPERIMENTAL DATA COLLETION MANAGEMENT
def init_data_collection(dir_path, DATAFIELDNAMES, filename="result_lassi-verify_pipeline_data.csv"):
    os.makedirs(dir_path, exist_ok=True)
    fp = os.path.join(dir_path, filename)
    if not os.path.exists(fp):
        with open(fp, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=DATAFIELDNAMES)
            writer.writeheader()
    return fp

_____
***STEP 2/2: Run the following block to implement the pipeline with the configuration parameters.***

In [5]:
# Experimental repeat for pass@k metrics
trial_number = 1
total_trials = 3

# Load prompt dictionary (to capture recent manual changes to script, if any)
prompts = PromptDictionary()

all_trials_start = time.time()

# Store the current home working directory (needed for code execution)
original_home_directory = os.getcwd()


# COLLECT THE EXPERIMENT CONFIGURATIONS
(
    app_folder_name, app_name, file_name, lines_of_code, source_lang, target_lang, model, model_coder, model_judge, num_ctx, num_ctx_coder,
    llm_source, use_context, use_context_summaries_only, few_shot_learning, run_code_analysis, context_knowledge_selection, source_folder, code_compiler,
    code_compiler_kwds, build_script, make_script, code_build_script, yes_execute, exe_input_parameters, experiment,
    display_response, test_with_error, generate_test_case, power_measure, yes_power_feedback, baseline_oneshot_goal,
    use_self_prompting, vary_llm_temp, verifyprompt, DATAFIELDNAMES, gpu_vendor,
    run_tier2, tier2_tolerance, run_tier3, tier3_tolerance,
    run_llm_judge
) = experimental_setup()

# IMPORT CONTEXT KNOWLEDGE FOR TARGET PROGRAMMING LANGUAGE
if gpu_vendor == "NVIDIA":
    input_context = "[BACKGROUND KNOWLEDGE: CUDA C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("cuda") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: CUDA C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]
elif gpu_vendor == "AMD":
    input_context = "[BACKGROUND KNOWLEDGE: HIP C++ PROGRAMMING BEST PRACTICES] " + prompts.get_codeknowledge("hip") + "\n\n"  # [/END GENERAL CUDA PROGRAMMING CONTEXT]
    input_context = input_context + "[BACKGROUND KNOWLEDGE: HIP C++ OPTIMIZATION GUIDE] " + prompts.get_codeknowledge(str(context_knowledge_selection)) + "\n\n"  #  [/END CUDA OPTIMIZATION CONTEXT]


# DEFINE PROMPT TEMPLATES
# TODO: MOST TASK PROMPT TEMPLATES WILL BE HANDLED BY THE PIPELINE METHODS
# If we are doing the runtime task, then set the system prompts, otherwise they are for ee
# we need to bring back in the translation systems prompt reference when we create LASSI-X
if (power_measure == False and yes_power_feedback == True):
    system_prompt_selection_index = "general_system_" + source_lang + "_runtime"
else:
    system_prompt_selection_index = source_lang + "_ee"  # "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
this_llm_system_prompt = prompts.get_system_prompt(str(system_prompt_selection_index))

coderefactor_prompt_selection_index = source_lang  # + "_to_" + target_lang  # Select which prompt to use from the prompt dictionary
if (not baseline_oneshot_goal and use_self_prompting):
    # TODO: THIS SHOULD BE A DEAD BRANCH BECAUSE WE DON'T WANT TO USE baseline_oneshot_goal ANYMORE
    his_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index))
else: 
    if (power_measure == False and yes_power_feedback == True):
        # RUNTIME TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_runtime"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext_runtime"))
    else:
        # EE TASK
        if use_context:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline"))
        else:
            this_llm_refactoredcode_prompt = prompts.get_code_ee_refactoring_prompt(str(coderefactor_prompt_selection_index + "_baseline_nocontext"))

# LOAD INPUT CODE TO REFACTOR
input_code_folder = source_folder + "/" + app_folder_name + "/"
if build_script:
    input_code_filename = app_name
elif make_script:
    input_code_filename = file_name
else:
    input_code_filename = app_folder_name + "-" + source_lang.lower() + "_main"
if source_lang == "OMP" or source_lang == "SYCL":
    input_code_extension = "cpp"
elif source_lang == "CUDA" or source_lang == "HIP":
    if (make_script and source_lang == "HIP"):
        input_code_extension = "cpp"  # custom for miniMDock and XSBench
    else:
        input_code_extension = "cu"
if make_script:
    source_file_directory = "generated_codes/" + input_code_folder
else:
    source_file_directory = "source_codes/" + input_code_folder
if make_script:
    # SELECT FILE TO REFACTOR from miniapp
    if source_lang == "CUDA":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/cuda/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    elif source_lang == "HIP":
        if app_name == "miniMDock":
            input_code_file_subdirectory = "device/hip/"
        elif app_name == "XSBench":
            input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_file_subdirectory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
else:
    input_code_file_subdirectory = ""
    source_file_path = source_file_directory + input_code_filename + "." + input_code_extension
    source_file_path = os.path.abspath(source_file_path)
with open(source_file_path, 'r') as file:
    input_code = file.read()
file.close()


# SPECIFY OUTPUT CODE SAVE
if build_script:
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
    output_code_file_name = app_name
elif make_script:
    output_code_file_name = file_name
    output_code_directory = "generated_codes/" + source_folder + "/" + app_folder_name
else:
    output_code_directory = "generated_codes/" + llm_source + "_" + model.replace('/', '--') + "/promptset_" + str(coderefactor_prompt_selection_index)
    output_code_file_name = input_code_filename + "_ee_refactorcode_" + source_lang + "_exp" + experiment

now = datetime.now(ZoneInfo("America/Chicago"))
dt_string = now.strftime("%m-%d-%Y_t%H_%M_%S")

# SPECIFIY DATA COLLECTION SAVE
experiment_data_directory = "generated_codes/experiment/" + experiment + "__" + dt_string
# Convert to absolute directory::
experiment_data_directory = os.path.abspath(experiment_data_directory)
experiment_data_file_object = init_data_collection(experiment_data_directory, DATAFIELDNAMES)
experiment_data = dict.fromkeys(DATAFIELDNAMES, None) # all keys start out as None
# Table 2 ★ per-attempt log — one row per code generation attempt (including self-correction retries)
attempt_log_file_path = init_data_collection(
    experiment_data_directory, lassi.pipeline_v4.ATTEMPT_LOG_FIELDNAMES,
    filename="result_lassi-verify_attempt_log.csv"
)
experiment_data["gpu_vendor"] = str(detected_vendor.upper())

while trial_number <= total_trials:
    # For the miniApp, we need to reset the source kernel file
    #  because the pipeline for this scenario must overwrite the 
    #  source code in the generated_codes folder.
    if make_script:
        if source_lang == "CUDA":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/cuda/kernels_orig_merged.cu"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main.cu"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/cuda/Main_orig_merged.cu"
        elif source_lang == "HIP":
            if app_name == "miniMDock":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/miniMDock/device/hip/kernels_orig_merged.cpp"
            elif app_name == "XSBench":
                working_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main.cpp"
                clean_kernel_file = "/home/cc/lassi/LASSI-EE/generated_codes/Miniapps/XSBench/hip/Main_orig_merged.cpp"
        try:
            print(f"[reset] Copying clean kernel -> working file\n"
                  f"        src: {clean_kernel_file}\n"
                  f"        dst: {working_kernel_file}")
            shutil.copyfile(clean_kernel_file, working_kernel_file)
            print("[reset] Success: working file reset to clean copy.")
        except FileNotFoundError as e:
            print(f"[reset] ERROR: file not found: {e.filename}")
        except PermissionError as e:
            print(f"[reset] ERROR: permission denied: {e.filename}")
        except OSError as e:
            # catches other IO errors (disk full, cross-device issues, etc.)
            print(f"[reset] ERROR: {e.__class__.__name__}: {e}")

    # CREATE METADATA RECORD CONTENT
    meta_data_file = "***** EXPERIMENT CONFIGURATION SUMMARY *****\n"
    meta_data_file += "\nGPU Vendor: " + str(detected_vendor.upper())
    meta_data_file += "\nDate-time: " + dt_string
    meta_data_file += "\nApplication: " + app_folder_name
    meta_data_file += "\n\nRefactoring task: " + system_prompt_selection_index
    meta_data_file += "\n\nLLM Source: " + llm_source
    meta_data_file += "\nModel: " + model
    meta_data_file += "\nModel coder: " + model_coder
    meta_data_file += "\nModel judge: " + model_judge
    meta_data_file += "\nSystem Prompt Index: " + str(coderefactor_prompt_selection_index)
    meta_data_file += "\nCode file template: " + output_code_directory + "/" + output_code_file_name
    meta_data_file += "\n\nLLM System Prompt: " + "\n\n" + this_llm_system_prompt
    # meta_data_file += "\n\nLLM Prompt for Code Generation: " + "\n\n" + this_llm_refactoredcode_prompt
    meta_data_file += "\n\nPower measurement: " + str(power_measure)
    
    # Display metadata
    print("\n" + meta_data_file + "\n")
    
    print("\n\n" + color.BLUE + color.BOLD + "***** BEGIN THE LASSI ENERGY EFFICIENCY PIPELINE (v4) *****" + color.END + " Trial #: " + str(trial_number) + "\n\n")
    
    # RUN LASSI-EE PIPELINE
    complete = lassi.pipeline_v4.auto_code_llmgeneration_pipeline(
        prompts,
        this_llm_system_prompt,
        this_llm_refactoredcode_prompt,
        llm_source, 
        model,
        model_coder,
        model_judge,
        num_ctx,
        num_ctx_coder,
        # arrayormatrix_size_range,
        # thread_count_range,
        use_context,
        use_context_summaries_only,
        few_shot_learning,
        run_code_analysis,
        input_context,
        input_code_file_subdirectory,
        input_code,
        lines_of_code,
        # target_code,
        target_lang,
        input_code_extension,
        source_lang,
        code_compiler,
        code_compiler_kwds,
        build_script,
        make_script,
        code_build_script,
        output_code_directory,
        output_code_file_name,
        source_file_path,
        display_response,
        test_with_error,
        generate_test_case,
        yes_execute,
        exe_input_parameters,
        meta_data_file,
        power_measure,
        yes_power_feedback,
        dt_string,
        baseline_oneshot_goal,
        use_self_prompting,
        vary_llm_temp,
        experiment,
        verifyprompt,
        trial_number,
        experiment_data_directory,
        experiment_data_file_object,
        experiment_data,
        app_name,
        run_tier2=run_tier2,
        tier2_tolerance=tier2_tolerance,
        run_tier3=run_tier3,
        tier3_tolerance=tier3_tolerance,
        run_llm_judge=run_llm_judge,
        attempt_log_file_path=attempt_log_file_path
    )
    
    # Reset to original working directory
    os.chdir(original_home_directory)
    
    if complete:
        print("\n\n" + color.GREEN + color.BOLD + "***** LASSI PIPELINE COMPLETED *****\n\n" + color.END)
    else:
        print("\n\n" + color.RED + color.BOLD + "***** LASSI PIPELINE FAILED TO COMPLETE *****\n\n" + color.END)

    # Increment experiment counter
    trial_number = trial_number + 1


all_trials_end = time.time() - all_trials_start

print("\n\n" + color.PURPLE + color.BOLD + "***** ALL TRIALS COMPLETED ***** Total time (min): " + str(round(all_trials_end/60, 2)) + "\n\n" + color.END)




***** EXPERIMENT CONFIGURATION SUMMARY *****

GPU Vendor: NVIDIA
Date-time: 03-17-2026_t09_55_08
Application: jaccard

Refactoring task: general_system_CUDA_runtime

LLM Source: argo
Model: gpt54
Model coder: gpt54
Model judge: gpt41
System Prompt Index: CUDA
Code file template: generated_codes/argo_gpt54/promptset_CUDA/jaccard-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA

LLM System Prompt: 

You are an expert C++/CUDA engineer. Your goal is to refactor provided CUDA C++ code to reduce wall-clock execution time by improving GPU and CPU utilization and parallelism, while preserving all original functionality and expected output of the provided code. You must also carefully read and incorporate the best practices and CUDA optimization techniques from the provided background documents and refactoring plans. You always surround your generated code with the three characters ```.

Power measurement: False



***** BEGIN THE LASSI ENERGY EFFICIE

Waiting for the system to settle down for 1 seconds...: 100%|██████████| 1/1 [00:01<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/jaccard/jaccard-cuda_main
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/jaccard
executable:  jaccard-cuda_main
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/source_codes/HeCBench/jaccard
Execute command: ['./jaccard-cuda_main', '1024', '512', '1000']

running: executable -  jaccard-cuda_main

----- < BEGIN: STANDARD OUTPUT FROM CODE EXECUTION > -----

Number of matrix rows and cols: 1024 512
Average execution time of kernels: 0.00544222 (s)
Average execution time of kernels: 0.00461005 (s)

THE RETURN CODE:  0

----- < END: Standard Output > -----


Program executed successfully!

----- END OF STAGE 0: TEST SOURCE -----

None

*** DEBUG: code_version: 0 | tracking_best_version: 0 | code_blocks len: 1 (next idx: 1) ***

----- STARTING CODE INSERT - Invariant Assertions -----


=====> VERIFY THE PRO

```cpp
/*
 * Copyright (c) 2019, NVIDIA CORPORATION.
 *
 * Licensed under the Apache License, Version 2.0 (the "License");
 * you may not use this file except in compliance with the License.
 * You may obtain a copy of the License at
 *
 *     http://www.apache.org/licenses/LICENSE-2.0
 *
 * Unless required by applicable law or agreed to in writing, software
 * distributed under the License is distributed on an "AS IS" BASIS,
 * WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 * See the License for the specific language governing permissions and
 * limitations under the License.
 */

#include <stdio.h>
#include <cstdio>
#include <algorithm>
#include <cmath>
#include <iostream>
#include <vector>
#include <chrono>
#include <cuda.h>
#include <cstdlib>

using namespace std;

#define MAX_KERNEL_THREADS 256
#define mask 0xFFFFFFFF

// float or double
typedef float vtype;
typedef vector<vector<vtype>> matrix;

template<typename T>
__device__
T parallel_prefix_sum(const int n, const int *ind, const T *w)
{

  T sum = 0.0;
  T last;

  int mn =(((n+blockDim.x-1)/blockDim.x)*blockDim.x); //n in multiple of blockDim.x
  for (int i=threadIdx.x; i<mn; i+=blockDim.x) {
    //All threads (especially the last one) must always participate
    //in the shfl instruction, otherwise their sum will be undefined.
    //So, the loop stopping condition is based on multiple of n in loop increments,
    //so that all threads enter into the loop and inside we make sure we do not
    //read out of bounds memory checking for the actual size n.

    //check if the thread is valid
    bool valid  = i<n;

    //Notice that the last thread is used to propagate the prefix sum.
    //For all the threads, in the first iteration the last is 0, in the following
    //iterations it is the value at the last thread of the previous iterations.

    //get the value of the last thread
    last = __shfl_sync(mask, sum, blockDim.x-1, blockDim.x);

    //if you are valid read the value from memory, otherwise set your value to 0
    sum = (valid) ? w[ind[i]] : 0.0;

    //do prefix sum (of size warpSize=blockDim.x =< 32)
    for (int j=1; j<blockDim.x; j*=2) {
      T v = __shfl_up_sync(mask, sum, j, blockDim.x);
      if (threadIdx.x >= j) sum += v;
    }
    //shift by last
    sum += last;
    //notice that no __threadfence or __syncthreads are needed in this implementation
  }
  //get the value of the last thread (to all threads)
  last = __shfl_sync(mask, sum, blockDim.x-1, blockDim.x);

  return last;
}

// Volume of neighboors (*weight_s)
template<bool weighted, typename T>
__global__ void
jaccard_row_sum(const int n,
                const int *__restrict__ csrPtr,
                const int *__restrict__ csrInd,
                const T *__restrict__ w,
                      T *__restrict__ work)
{
  for (int row=threadIdx.y+blockIdx.y*blockDim.y; row<n; row+=gridDim.y*blockDim.y) {
    int start = csrPtr[row];
    int end   = csrPtr[row+1];
    int length= end-start;
    //compute row sums
    if (weighted) {
      T sum = parallel_prefix_sum(length, csrInd + start, w);
      if (threadIdx.x == 0) work[row] = sum;
    }
    else {
      work[row] = (T)length;
    }
  }
}

// Volume of intersections (*weight_i) and cumulated volume of neighboors (*weight_s)
// Note the number of columns is constrained by the number of rows
template<bool weighted, typename T>
__global__ void
jaccard_is(const int n, const int e,
           const int *__restrict__ csrPtr,
           const int *__restrict__ csrInd,
           const T *__restrict__ v,
           const T *__restrict__ work,
                 T *__restrict__ weight_i,
                 T *__restrict__ weight_s)
{
  for (int row=threadIdx.z+blockIdx.z*blockDim.z; row<n; row+=gridDim.z*blockDim.z) {
    for (int j=csrPtr[row]+threadIdx.y+blockIdx.y*blockDim.y;
             j<csrPtr[row+1]; j+=gridDim.y*blockDim.y) {
      int col = csrInd[j];
      //find which row has least elements (and call it reference row)
      int Ni = csrPtr[row+1] - csrPtr[row];
      int Nj = csrPtr[col+1] - csrPtr[col];
      int ref= (Ni < Nj) ? row : col;
      int cur= (Ni < Nj) ? col : row;

      //compute new sum weights
      weight_s[j] = work[row] + work[col];

      //compute new intersection weights
      //search for the element with the same column index in the reference row
      for (int i=csrPtr[ref]+threadIdx.x+blockIdx.x*blockDim.x; i<csrPtr[ref+1]; i+=gridDim.x*blockDim.x) {
        int match  =-1;
        int ref_col = csrInd[i];
        T ref_val = weighted ? v[ref_col] : (T)1.0;

        //binary search (column indices are sorted within each row)
        int left = csrPtr[cur];
        int right= csrPtr[cur+1]-1;
        while(left <= right){
          int middle = (left+right)>>1;
          int cur_col= csrInd[middle];
          if (cur_col > ref_col) {
            right=middle-1;
          }
          else if (cur_col < ref_col) {
            left=middle+1;
          }
          else {
            match = middle;
            break;
          }
        }

        //if the element with the same column index in the reference row has been found
        if (match != -1){
          atomicAdd(&weight_i[j],ref_val);
        }
      }
    }
  }
}

// Reference https://github.com/SPEAR-UIC/CodeGreen/tree/main/lassi_solutions
template<bool weighted, typename T>
__global__ void
jaccard_is_opt(const int n, const int e,
               const int *__restrict__ csrPtr,
               const int *__restrict__ csrInd,
               const T *__restrict__ v,
               const T *__restrict__ work,
                     T *__restrict__ weight_i,
                     T *__restrict__ weight_s)
{
  for (int row=threadIdx.z+blockIdx.z*blockDim.z; row<n; row+=gridDim.z*blockDim.z) {
    for (int j=csrPtr[row]+threadIdx.y+blockIdx.y*blockDim.y;
             j<csrPtr[row+1]; j+=gridDim.y*blockDim.y) {
      int col = csrInd[j];
      //find which row has least elements (and call it reference row)
      int Ni = csrPtr[row+1] - csrPtr[row];
      int Nj = csrPtr[col+1] - csrPtr[col];
      int ref= (Ni < Nj) ? row : col;
      int cur= (Ni < Nj) ? col : row;

      //compute new sum weights
      weight_s[j] = work[row] + work[col];

      //compute new intersection weights
      //search for the element with the same column index in the reference row
      if (threadIdx.x == 0) {
        T local_sum = 0;
        int i_ptr = csrPtr[ref];      // pointer in reference row
        int j_ptr = csrPtr[cur];        // pointer in current row
        int ref_end = csrPtr[ref+1];
        int cur_end = csrPtr[cur+1];

        // Two-pointer merge for intersection of the two sorted lists
        while (i_ptr < ref_end && j_ptr < cur_end) {
          int ref_col = csrInd[i_ptr];
          int cur_col = csrInd[j_ptr];
          if (ref_col == cur_col) {
            T ref_val = weighted ? v[ref_col] : (T)1.0;
            local_sum += ref_val;
            i_ptr++;
            j_ptr++;
          } else if (ref_col < cur_col) {
            i_ptr++;
          } else {
            j_ptr++;
          }
        }
        // perform a single atomic update per this j index
        if (local_sum != 0)
          atomicAdd(&weight_i[j], local_sum);
      }
    }
  }
}

template<bool weighted, typename T>
__global__ void
jaccard_jw(const int e,
    const T *__restrict__ csrVal,
    const T gamma,
    const T *__restrict__ weight_i,
    const T *__restrict__ weight_s,
          T *__restrict__ weight_j)
{
  for (int j=threadIdx.x+blockIdx.x*blockDim.x; j<e; j+=gridDim.x*blockDim.x) {
    T Wi =  weight_i[j];
    T Ws =  weight_s[j];
    weight_j[j] = (gamma*csrVal[j])* (Wi/(Ws-Wi));
  }
}

template <bool weighted, typename T>
__global__ void
fill(const int e, T* w, const T value)
{
  for (int j=threadIdx.x+blockIdx.x*blockDim.x; j<e; j+=gridDim.x*blockDim.x) {
    // e.g. w[0] is the weight of a non-zeron element when csr_ind[i] equals 0.
    // So multiple non-zero elements on different rows of a matrix may share
    // the same weight value
    w[j] = weighted ? (T)(j+1)/e : value;
  }
}

template <bool weighted, typename T>
void jaccard_weight (const int iteration, const int n, const int e,
    int* csr_ptr, int* csr_ind, T* csr_val)
{
  const T gamma = (T)0.46;  // arbitrary

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate input pointers and scalar parameters at function entry
  {
    bool _t1_pass = (csr_ptr != NULL && csr_ind != NULL && csr_val != NULL &&
                     iteration > 0 && n > 0 && e >= 0 && std::isfinite((double)gamma));
    printf("LASSI-VERIFY ASSERTIONS [jw_entry_params_valid_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate CSR pointer structure at function entry
  {
    bool _t1_pass = true;
    if (csr_ptr == NULL || n <= 0) {
      _t1_pass = false;
    } else {
      if (csr_ptr[0] != 0) _t1_pass = false;
      for (int i = 0; i < n && _t1_pass; i++) {
        if (!(csr_ptr[i] <= csr_ptr[i+1])) { _t1_pass = false; break; }
      }
      if (_t1_pass && csr_ptr[n] != e) _t1_pass = false;
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_csr_ptr_valid_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate CSR indices and values before host-to-device transfer
  {
    bool _t1_pass = true;
    if (csr_ind == NULL || csr_val == NULL || e < 0) {
      _t1_pass = false;
    } else {
      for (int i = 0; i < e; i++) {
        if (!(csr_ind[i] >= 0 && csr_ind[i] < n)) { _t1_pass = false; break; }
        if (!std::isfinite((double)csr_val[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_csr_inputs_finite_bounds_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  T *d_weight_i,
    *d_weight_s,
    *d_weight_j,
    *d_work;
  int *d_csrInd;
  int *d_csrPtr;
  T *d_csrVal;

#ifdef DEBUG
  T* weight_i = (T*) malloc (sizeof(T) * e);
  T* weight_s = (T*) malloc (sizeof(T) * e);
  T* work = (T*) malloc (sizeof(T) * n);
#endif
  T* weight_j = (T*) malloc (sizeof(T) * e);

  cudaMalloc ((void**)&d_work, sizeof(T) * n);
  cudaMalloc ((void**)&d_weight_i, sizeof(T) * e);
  cudaMalloc ((void**)&d_weight_s, sizeof(T) * e);
  cudaMalloc ((void**)&d_weight_j, sizeof(T) * e);
  cudaMalloc ((void**)&d_csrVal, sizeof(T) * e);
  cudaMalloc ((void**)&d_csrPtr, sizeof(int) * (n+1));
  cudaMalloc ((void**)&d_csrInd, sizeof(int) * e);

  cudaMemcpy(d_csrPtr, csr_ptr, sizeof(int) * (n+1), cudaMemcpyHostToDevice);
  cudaMemcpy(d_csrInd, csr_ind, sizeof(int) * e, cudaMemcpyHostToDevice);
  cudaMemcpy(d_csrVal, csr_val, sizeof(T) * e, cudaMemcpyHostToDevice);

  cudaDeviceSynchronize();
  auto start = std::chrono::steady_clock::now();

  for (int i = 0; i < iteration; i++) {
    dim3 nthreads, nblocks; // reuse for multiple kernels

    nthreads.x = MAX_KERNEL_THREADS;
    nthreads.y = 1;
    nthreads.z = 1;
    nblocks.x  = (e+MAX_KERNEL_THREADS-1) / MAX_KERNEL_THREADS;
    nblocks.y  = 1;
    nblocks.z  = 1;

    fill<weighted, T><<<nblocks, nthreads>>>(e, d_weight_j, (T)1.0);
#ifdef DEBUG
    cudaMemcpy(weight_j, d_weight_j, sizeof(T) * e, cudaMemcpyDeviceToHost);
    for (int i = 0; i < e; i++) printf("wj: %d %f\n", i, weight_j[i]);
#endif

    // initialize volume of intersections
    fill<false, T><<<nblocks, nthreads>>>(e, d_weight_i, (T)0.0);

    // compute row sum with prefix sum
    const int y = 4;
    nthreads.x = 64/y;
    nthreads.y = y;
    nthreads.z = 1;
    nblocks.x  = 1;
    nblocks.y  = (n + nthreads.y - 1) / nthreads.y;  // less than MAX CUDA BLOCKs
    nblocks.z  = 1;
    jaccard_row_sum<weighted,T><<<nblocks,nthreads>>>(n, d_csrPtr, d_csrInd, d_weight_j, d_work);

#ifdef DEBUG
    cudaMemcpy(work, d_work, sizeof(T) * n, cudaMemcpyDeviceToHost);
    for (int i = 0; i < n; i++) printf("work: %d %f\n", i, work[i]);
#endif

    // compute volume of intersections (*weight_i) and cumulated volume of neighboors (*weight_s)
    // nthreads.x * nthreads.y * nthreads.z <= 256
    nthreads.x = 32/y;
    nthreads.y = y;
    nthreads.z = 8;
    nblocks.x  = 1;
    nblocks.y  = 1;
    nblocks.z  = (n + nthreads.z - 1)/nthreads.z; // less than CUDA_MAX_BLOCKS);
    jaccard_is_opt<weighted,T><<<nblocks,nthreads>>>(n, e, d_csrPtr,
        d_csrInd, d_weight_j, d_work, d_weight_i, d_weight_s);

#ifdef DEBUG
    cudaMemcpy(weight_i, d_weight_i, sizeof(T) * e, cudaMemcpyDeviceToHost);
    cudaMemcpy(weight_s, d_weight_s, sizeof(T) * e, cudaMemcpyDeviceToHost);
    for (int i = 0; i < e; i++) printf("wi: %d %f\n", i, weight_i[i]);
    for (int i = 0; i < e; i++) printf("ws: %d %f\n", i, weight_s[i]);
#endif

    // compute jaccard weights
    nthreads.x = std::min(e, MAX_KERNEL_THREADS);
    nthreads.y = 1;
    nthreads.z = 1;
    nblocks.x  = (e + nthreads.x - 1)/nthreads.x;  // less than MAX CUDA BLOCKs
    nblocks.y  = 1;
    nblocks.z  = 1;
    jaccard_jw<weighted,T><<<nblocks,nthreads>>>(e,
        d_csrVal, gamma, d_weight_i, d_weight_s, d_weight_j);
  }

  cudaDeviceSynchronize();
  auto end = std::chrono::steady_clock::now();
  auto time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  cout << "Average execution time of kernels: " << (time * 1e-9f) / iteration << " (s)\n";

  cudaMemcpy(weight_j, d_weight_j, sizeof(T) * e, cudaMemcpyDeviceToHost);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate output weights after device-to-host transfer are finite
  {
    bool _t1_pass = true;
    if (weight_j == NULL || e < 0) {
      _t1_pass = false;
    } else {
      for (int i = 0; i < e; i++) {
        if (!std::isfinite((double)weight_j[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_output_finite_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate final result size-related properties at function exit
  {
    bool _t1_pass = (e == 0 || weight_j != NULL);
    printf("LASSI-VERIFY ASSERTIONS [jw_exit_result_valid_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

#ifdef DEBUG
  // verify using known values when weighted is true
  float error;

  if (weighted)
    error = std::fabs(weight_j[0] - 0.306667) +
            std::fabs(weight_j[1] - 0.000000) +
            std::fabs(weight_j[2] - 3.680000) +
            std::fabs(weight_j[3] - 1.380000) +
            std::fabs(weight_j[4] - 0.788571) +
            std::fabs(weight_j[5] - 0.460000);

  else
    error = std::fabs(weight_j[0] - 0.230000) +
            std::fabs(weight_j[1] - 0.000000) +
            std::fabs(weight_j[2] - 3.680000) +
            std::fabs(weight_j[3] - 1.380000) +
            std::fabs(weight_j[4] - 0.920000) +
            std::fabs(weight_j[5] - 0.460000);

  if (error > 1e-5) {
    for (int i = 0; i < e; i++) printf("wj: %d %f\n", i, weight_j[i]);
    printf("FAIL");
  } else {
    printf("PASS");
  }
  printf("\n");
#endif

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaFree (d_work);
  cudaFree (d_weight_i);
  cudaFree (d_weight_s);
  cudaFree (d_weight_j);
  cudaFree (d_csrInd);
  cudaFree (d_csrVal);
  cudaFree (d_csrPtr);
  free(weight_j);
#ifdef DEBUG
  free(weight_i);
  free(weight_s);
  free(work);
#endif
}

// Utilities
void printMatrix(const matrix& M)
{
  int m = M.size();
  int n = M[0].size();
  for (int i = 0; i < m; i++) {
    for (int j = 0; j < n; j++)
      cout << M[i][j] << " ";
    cout << endl;
  }
}

template <typename T>
void printVector(const vector<T>& V, char* msg)
{
  cout << msg << "[ ";
  for_each(V.begin(), V.end(), [](int a) { cout << a << " "; });
  cout << "]" << endl;
}

// Reference: https://www.geeksforgeeks.org/sparse-matrix-representations-set-3-csr/
int main(int argc, char** argv)
{
  int iteration = 10;

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate command-line configuration at program entry
  {
#ifdef DEBUG
    bool _t1_pass = true;
#else
    bool _t1_pass = (argc >= 4);
#endif
    printf("LASSI-VERIFY ASSERTIONS [main_args_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

#ifdef DEBUG
  matrix M  = {
    { 0, 0, 0, 1},
    { 5, 8, 0, 0},
    { 0, 0, 3, 0},
    { 0, 6, 0, 1}
  };
#else

  int numRow = atoi(argv[1]);
  int numCol = atoi(argv[2]);
  iteration = atoi(argv[3]);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate parsed matrix dimensions and iteration count
  {
    bool _t1_pass = (numRow > 0 && numCol > 0 && iteration > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_dims_iteration_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  srand(2);

  matrix M;
  vector<vtype> rowElems(numCol);
  for (int r = 0; r < numRow; r++) {
    for (int c = 0; c < numCol; c++)
      rowElems[c] = rand() % 10;
    M.push_back(rowElems);
  }
#endif

  int row = M.size();
  int col = M[0].size();

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate constructed matrix dimensions and row consistency
  {
    bool _t1_pass = (row > 0 && col > 0);
    if (_t1_pass) {
      for (int i = 0; i < row; i++) {
        if ((int)M[i].size() != col) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_matrix_shape_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate matrix entries are finite before CSR conversion
  {
    bool _t1_pass = true;
    for (int i = 0; i < row && _t1_pass; i++) {
      for (int j = 0; j < col; j++) {
        if (!std::isfinite((double)M[i][j])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_matrix_finite]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  printf("Number of matrix rows and cols: %d %d\n", row, col);
  vector<vtype> csr_val;
  vector<int> csr_ptr = { 0 }; // require -std=c++11
  vector<int> csr_ind;
  int nnz = 0; // count Number of non-zero elements in each row

  for (int i = 0; i < row; i++) {
    for (int j = 0; j < col; j++) {
      if (M[i][j] != (vtype)0) {
        csr_val.push_back(M[i][j]);
        csr_ind.push_back(j);
        nnz++;
      }
    }
    csr_ptr.push_back(nnz);
  }

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate CSR arrays after construction and before GPU use
  {
    bool _t1_pass = true;
    if ((int)csr_ptr.size() != row + 1) _t1_pass = false;
    if (_t1_pass && (int)csr_ind.size() != nnz) _t1_pass = false;
    if (_t1_pass && (int)csr_val.size() != nnz) _t1_pass = false;
    if (_t1_pass && csr_ptr[0] != 0) _t1_pass = false;
    if (_t1_pass && csr_ptr[row] != nnz) _t1_pass = false;
    for (int i = 0; i < row && _t1_pass; i++) {
      if (!(csr_ptr[i] <= csr_ptr[i+1])) { _t1_pass = false; break; }
    }
    for (int i = 0; i < nnz && _t1_pass; i++) {
      if (!(csr_ind[i] >= 0 && csr_ind[i] < col)) { _t1_pass = false; break; }
      if (!std::isfinite((double)csr_val[i])) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_csr_valid_before_gpu]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // print when the matrix is small
  if (row <= 16 && col <= 16) {
    printMatrix(M);
    printVector(csr_val, (char*)"values = ");
    printVector(csr_ptr, (char*)"row pointer = ");
    printVector(csr_ind, (char*)"col indices = ");
  }

  jaccard_weight<true, vtype>(iteration, row, nnz, csr_ptr.data(), csr_ind.data(), csr_val.data());
  jaccard_weight<false, vtype>(iteration, row, nnz, csr_ptr.data(), csr_ind.data(), csr_val.data());

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate final host-side structural properties at program exit
  {
    bool _t1_pass = (row > 0 && col > 0 && nnz >= 0 && csr_ptr[row] == nnz);
    printf("LASSI-VERIFY ASSERTIONS [main_exit_structure_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  return 0;
}
```


----- STARTING CODE INSERT - Passthrough Data Dump Blocks -----


=====> VERIFY THE PROMPTING <=====

TEMPERATURE: 0.2

SYSTEM: You are an expert C++/CUDA engineer. Your goal is to refactor provided CUDA C++ code to reduce wall-clock execution time by improving GPU and CPU utilization and parallelism, while preserving all original functionality and expected output. You always surround your generated code with the three characters ```.

PROMPT: 
        You are preparing GPU-accelerated scientific code for Tier 2 empirical equivalence checking
        as part of the LASSI-Verify verification framework.

        Your task is to instrument the source code with **passthrough data blocks** at key observation
        points. These blocks capture computational state to stdout in a structured, labeled format.
        Both the source (SOURCE_verify) and every LASSI-generated refactored version will carry these
        blocks unaltered, enabling deterministic comparison of observable output at 

```cpp
/*
 * Copyright (c) 2019, NVIDIA CORPORATION.
 *
 * Licensed under the Apache License, Version 2.0 (the "License");
 * you may not use this file except in compliance with the License.
 * You may obtain a copy of the License at
 *
 *     http://www.apache.org/licenses/LICENSE-2.0
 *
 * Unless required by applicable law or agreed to in writing, software
 * distributed under the License is distributed on an "AS IS" BASIS,
 * WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 * See the License for the specific language governing permissions and
 * limitations under the License.
 */

#include <stdio.h>
#include <cstdio>
#include <algorithm>
#include <cmath>
#include <iostream>
#include <vector>
#include <chrono>
#include <cuda.h>
#include <cstdlib>

using namespace std;

#define MAX_KERNEL_THREADS 256
#define mask 0xFFFFFFFF

// float or double
typedef float vtype;
typedef vector<vector<vtype>> matrix;

template<typename T>
__device__
T parallel_prefix_sum(const int n, const int *ind, const T *w)
{

  T sum = 0.0;
  T last;

  int mn =(((n+blockDim.x-1)/blockDim.x)*blockDim.x); //n in multiple of blockDim.x
  for (int i=threadIdx.x; i<mn; i+=blockDim.x) {
    //All threads (especially the last one) must always participate
    //in the shfl instruction, otherwise their sum will be undefined.
    //So, the loop stopping condition is based on multiple of n in loop increments,
    //so that all threads enter into the loop and inside we make sure we do not
    //read out of bounds memory checking for the actual size n.

    //check if the thread is valid
    bool valid  = i<n;

    //Notice that the last thread is used to propagate the prefix sum.
    //For all the threads, in the first iteration the last is 0, in the following
    //iterations it is the value at the last thread of the previous iterations.

    //get the value of the last thread
    last = __shfl_sync(mask, sum, blockDim.x-1, blockDim.x);

    //if you are valid read the value from memory, otherwise set your value to 0
    sum = (valid) ? w[ind[i]] : 0.0;

    //do prefix sum (of size warpSize=blockDim.x =< 32)
    for (int j=1; j<blockDim.x; j*=2) {
      T v = __shfl_up_sync(mask, sum, j, blockDim.x);
      if (threadIdx.x >= j) sum += v;
    }
    //shift by last
    sum += last;
    //notice that no __threadfence or __syncthreads are needed in this implementation
  }
  //get the value of the last thread (to all threads)
  last = __shfl_sync(mask, sum, blockDim.x-1, blockDim.x);

  return last;
}

// Volume of neighboors (*weight_s)
template<bool weighted, typename T>
__global__ void
jaccard_row_sum(const int n,
                const int *__restrict__ csrPtr,
                const int *__restrict__ csrInd,
                const T *__restrict__ w,
                      T *__restrict__ work)
{
  for (int row=threadIdx.y+blockIdx.y*blockDim.y; row<n; row+=gridDim.y*blockDim.y) {
    int start = csrPtr[row];
    int end   = csrPtr[row+1];
    int length= end-start;
    //compute row sums
    if (weighted) {
      T sum = parallel_prefix_sum(length, csrInd + start, w);
      if (threadIdx.x == 0) work[row] = sum;
    }
    else {
      work[row] = (T)length;
    }
  }
}

// Volume of intersections (*weight_i) and cumulated volume of neighboors (*weight_s)
// Note the number of columns is constrained by the number of rows
template<bool weighted, typename T>
__global__ void
jaccard_is(const int n, const int e,
           const int *__restrict__ csrPtr,
           const int *__restrict__ csrInd,
           const T *__restrict__ v,
           const T *__restrict__ work,
                 T *__restrict__ weight_i,
                 T *__restrict__ weight_s)
{
  for (int row=threadIdx.z+blockIdx.z*blockDim.z; row<n; row+=gridDim.z*blockDim.z) {
    for (int j=csrPtr[row]+threadIdx.y+blockIdx.y*blockDim.y;
             j<csrPtr[row+1]; j+=gridDim.y*blockDim.y) {
      int col = csrInd[j];
      //find which row has least elements (and call it reference row)
      int Ni = csrPtr[row+1] - csrPtr[row];
      int Nj = csrPtr[col+1] - csrPtr[col];
      int ref= (Ni < Nj) ? row : col;
      int cur= (Ni < Nj) ? col : row;

      //compute new sum weights
      weight_s[j] = work[row] + work[col];

      //compute new intersection weights
      //search for the element with the same column index in the reference row
      for (int i=csrPtr[ref]+threadIdx.x+blockIdx.x*blockDim.x; i<csrPtr[ref+1]; i+=gridDim.x*blockDim.x) {
        int match  =-1;
        int ref_col = csrInd[i];
        T ref_val = weighted ? v[ref_col] : (T)1.0;

        //binary search (column indices are sorted within each row)
        int left = csrPtr[cur];
        int right= csrPtr[cur+1]-1;
        while(left <= right){
          int middle = (left+right)>>1;
          int cur_col= csrInd[middle];
          if (cur_col > ref_col) {
            right=middle-1;
          }
          else if (cur_col < ref_col) {
            left=middle+1;
          }
          else {
            match = middle;
            break;
          }
        }

        //if the element with the same column index in the reference row has been found
        if (match != -1){
          atomicAdd(&weight_i[j],ref_val);
        }
      }
    }
  }
}

// Reference https://github.com/SPEAR-UIC/CodeGreen/tree/main/lassi_solutions
template<bool weighted, typename T>
__global__ void
jaccard_is_opt(const int n, const int e,
               const int *__restrict__ csrPtr,
               const int *__restrict__ csrInd,
               const T *__restrict__ v,
               const T *__restrict__ work,
                     T *__restrict__ weight_i,
                     T *__restrict__ weight_s)
{
  for (int row=threadIdx.z+blockIdx.z*blockDim.z; row<n; row+=gridDim.z*blockDim.z) {
    for (int j=csrPtr[row]+threadIdx.y+blockIdx.y*blockDim.y;
             j<csrPtr[row+1]; j+=gridDim.y*blockDim.y) {
      int col = csrInd[j];
      //find which row has least elements (and call it reference row)
      int Ni = csrPtr[row+1] - csrPtr[row];
      int Nj = csrPtr[col+1] - csrPtr[col];
      int ref= (Ni < Nj) ? row : col;
      int cur= (Ni < Nj) ? col : row;

      //compute new sum weights
      weight_s[j] = work[row] + work[col];

      //compute new intersection weights
      //search for the element with the same column index in the reference row
      if (threadIdx.x == 0) {
        T local_sum = 0;
        int i_ptr = csrPtr[ref];      // pointer in reference row
        int j_ptr = csrPtr[cur];        // pointer in current row
        int ref_end = csrPtr[ref+1];
        int cur_end = csrPtr[cur+1];

        // Two-pointer merge for intersection of the two sorted lists
        while (i_ptr < ref_end && j_ptr < cur_end) {
          int ref_col = csrInd[i_ptr];
          int cur_col = csrInd[j_ptr];
          if (ref_col == cur_col) {
            T ref_val = weighted ? v[ref_col] : (T)1.0;
            local_sum += ref_val;
            i_ptr++;
            j_ptr++;
          } else if (ref_col < cur_col) {
            i_ptr++;
          } else {
            j_ptr++;
          }
        }
        // perform a single atomic update per this j index
        if (local_sum != 0)
          atomicAdd(&weight_i[j], local_sum);
      }
    }
  }
}

template<bool weighted, typename T>
__global__ void
jaccard_jw(const int e,
    const T *__restrict__ csrVal,
    const T gamma,
    const T *__restrict__ weight_i,
    const T *__restrict__ weight_s,
          T *__restrict__ weight_j)
{
  for (int j=threadIdx.x+blockIdx.x*blockDim.x; j<e; j+=gridDim.x*blockDim.x) {
    T Wi =  weight_i[j];
    T Ws =  weight_s[j];
    weight_j[j] = (gamma*csrVal[j])* (Wi/(Ws-Wi));
  }
}

template <bool weighted, typename T>
__global__ void
fill(const int e, T* w, const T value)
{
  for (int j=threadIdx.x+blockIdx.x*blockDim.x; j<e; j+=gridDim.x*blockDim.x) {
    // e.g. w[0] is the weight of a non-zeron element when csr_ind[i] equals 0.
    // So multiple non-zero elements on different rows of a matrix may share
    // the same weight value
    w[j] = weighted ? (T)(j+1)/e : value;
  }
}

template <bool weighted, typename T>
void jaccard_weight (const int iteration, const int n, const int e,
    int* csr_ptr, int* csr_ind, T* csr_val)
{
  const T gamma = (T)0.46;  // arbitrary

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate input pointers and scalar parameters at function entry
  {
    bool _t1_pass = (csr_ptr != NULL && csr_ind != NULL && csr_val != NULL &&
                     iteration > 0 && n > 0 && e >= 0 && std::isfinite((double)gamma));
    printf("LASSI-VERIFY ASSERTIONS [jw_entry_params_valid_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate CSR pointer structure at function entry
  {
    bool _t1_pass = true;
    if (csr_ptr == NULL || n <= 0) {
      _t1_pass = false;
    } else {
      if (csr_ptr[0] != 0) _t1_pass = false;
      for (int i = 0; i < n && _t1_pass; i++) {
        if (!(csr_ptr[i] <= csr_ptr[i+1])) { _t1_pass = false; break; }
      }
      if (_t1_pass && csr_ptr[n] != e) _t1_pass = false;
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_csr_ptr_valid_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate CSR indices and values before host-to-device transfer
  {
    bool _t1_pass = true;
    if (csr_ind == NULL || csr_val == NULL || e < 0) {
      _t1_pass = false;
    } else {
      for (int i = 0; i < e; i++) {
        if (!(csr_ind[i] >= 0 && csr_ind[i] < n)) { _t1_pass = false; break; }
        if (!std::isfinite((double)csr_val[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_csr_inputs_finite_bounds_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  T *d_weight_i,
    *d_weight_s,
    *d_weight_j,
    *d_work;
  int *d_csrInd;
  int *d_csrPtr;
  T *d_csrVal;

#ifdef DEBUG
  T* weight_i = (T*) malloc (sizeof(T) * e);
  T* weight_s = (T*) malloc (sizeof(T) * e);
  T* work = (T*) malloc (sizeof(T) * n);
#endif
  T* weight_j = (T*) malloc (sizeof(T) * e);

  cudaMalloc ((void**)&d_work, sizeof(T) * n);
  cudaMalloc ((void**)&d_weight_i, sizeof(T) * e);
  cudaMalloc ((void**)&d_weight_s, sizeof(T) * e);
  cudaMalloc ((void**)&d_weight_j, sizeof(T) * e);
  cudaMalloc ((void**)&d_csrVal, sizeof(T) * e);
  cudaMalloc ((void**)&d_csrPtr, sizeof(int) * (n+1));
  cudaMalloc ((void**)&d_csrInd, sizeof(int) * e);

  cudaMemcpy(d_csrPtr, csr_ptr, sizeof(int) * (n+1), cudaMemcpyHostToDevice);
  cudaMemcpy(d_csrInd, csr_ind, sizeof(int) * e, cudaMemcpyHostToDevice);
  cudaMemcpy(d_csrVal, csr_val, sizeof(T) * e, cudaMemcpyHostToDevice);

  cudaDeviceSynchronize();
  auto start = std::chrono::steady_clock::now();

  for (int i = 0; i < iteration; i++) {
    dim3 nthreads, nblocks; // reuse for multiple kernels

    nthreads.x = MAX_KERNEL_THREADS;
    nthreads.y = 1;
    nthreads.z = 1;
    nblocks.x  = (e+MAX_KERNEL_THREADS-1) / MAX_KERNEL_THREADS;
    nblocks.y  = 1;
    nblocks.z  = 1;

    fill<weighted, T><<<nblocks, nthreads>>>(e, d_weight_j, (T)1.0);
#ifdef DEBUG
    cudaMemcpy(weight_j, d_weight_j, sizeof(T) * e, cudaMemcpyDeviceToHost);
    for (int i = 0; i < e; i++) printf("wj: %d %f\n", i, weight_j[i]);
#endif

    // initialize volume of intersections
    fill<false, T><<<nblocks, nthreads>>>(e, d_weight_i, (T)0.0);

    // compute row sum with prefix sum
    const int y = 4;
    nthreads.x = 64/y;
    nthreads.y = y;
    nthreads.z = 1;
    nblocks.x  = 1;
    nblocks.y  = (n + nthreads.y - 1) / nthreads.y;  // less than MAX CUDA BLOCKs
    nblocks.z  = 1;
    jaccard_row_sum<weighted,T><<<nblocks,nthreads>>>(n, d_csrPtr, d_csrInd, d_weight_j, d_work);

#ifdef DEBUG
    cudaMemcpy(work, d_work, sizeof(T) * n, cudaMemcpyDeviceToHost);
    for (int i = 0; i < n; i++) printf("work: %d %f\n", i, work[i]);
#endif

    // compute volume of intersections (*weight_i) and cumulated volume of neighboors (*weight_s)
    // nthreads.x * nthreads.y * nthreads.z <= 256
    nthreads.x = 32/y;
    nthreads.y = y;
    nthreads.z = 8;
    nblocks.x  = 1;
    nblocks.y  = 1;
    nblocks.z  = (n + nthreads.z - 1)/nthreads.z; // less than CUDA_MAX_BLOCKS);
    jaccard_is_opt<weighted,T><<<nblocks,nthreads>>>(n, e, d_csrPtr,
        d_csrInd, d_weight_j, d_work, d_weight_i, d_weight_s);

#ifdef DEBUG
    cudaMemcpy(weight_i, d_weight_i, sizeof(T) * e, cudaMemcpyDeviceToHost);
    cudaMemcpy(weight_s, d_weight_s, sizeof(T) * e, cudaMemcpyDeviceToHost);
    for (int i = 0; i < e; i++) printf("wi: %d %f\n", i, weight_i[i]);
    for (int i = 0; i < e; i++) printf("ws: %d %f\n", i, weight_s[i]);
#endif

    // compute jaccard weights
    nthreads.x = std::min(e, MAX_KERNEL_THREADS);
    nthreads.y = 1;
    nthreads.z = 1;
    nblocks.x  = (e + nthreads.x - 1)/nthreads.x;  // less than MAX CUDA BLOCKs
    nblocks.y  = 1;
    nblocks.z  = 1;
    jaccard_jw<weighted,T><<<nblocks,nthreads>>>(e,
        d_csrVal, gamma, d_weight_i, d_weight_s, d_weight_j);
  }

  cudaDeviceSynchronize();
  auto end = std::chrono::steady_clock::now();
  auto time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  cout << "Average execution time of kernels: " << (time * 1e-9f) / iteration << " (s)\n";

  cudaMemcpy(weight_j, d_weight_j, sizeof(T) * e, cudaMemcpyDeviceToHost);

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [final_weight_j] (do not edit or delete this block) >>>
  {
    // --- LASSI-VERIFY PASSTHROUGH: capture final output buffer for empirical equivalence ---
    T* lv_host_buf = (T*) malloc(e * sizeof(T));
    cudaMemcpy(lv_host_buf, d_weight_j, e * sizeof(T), cudaMemcpyDeviceToHost);

    T lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
    double lv_sum = 0.0;
    for (int lv_i = 0; lv_i < e; lv_i++) {
      if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
      if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
      lv_sum += (double)lv_host_buf[lv_i];
    }
    double lv_mean = lv_sum / e;

    printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [final_weight_j] ===\n");
    printf("Buffer: d_weight_j\n");
    printf("Type: float\n");
    printf("Elements: %d\n", (int)e);
    printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
           (double)lv_min, (double)lv_max, lv_mean);

    int lv_sample = (e < 10) ? (int)e : 10;
    printf("Sample_first10:");
    for (int lv_i = 0; lv_i < lv_sample; lv_i++)
      printf(" %.17g", (double)lv_host_buf[lv_i]);
    printf("\n");

    if (e > 10) {
      printf("Sample_last10:");
      for (int lv_i = (int)e - 10; lv_i < (int)e; lv_i++)
        printf(" %.17g", (double)lv_host_buf[lv_i]);
      printf("\n");
    }
    printf("=== END LASSI-VERIFY PASSTHROUGH [final_weight_j] ===\n");
    free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate output weights after device-to-host transfer are finite
  {
    bool _t1_pass = true;
    if (weight_j == NULL || e < 0) {
      _t1_pass = false;
    } else {
      for (int i = 0; i < e; i++) {
        if (!std::isfinite((double)weight_j[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_output_finite_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate final result size-related properties at function exit
  {
    bool _t1_pass = (e == 0 || weight_j != NULL);
    printf("LASSI-VERIFY ASSERTIONS [jw_exit_result_valid_%s]: %s\n", weighted ? "weighted" : "unweighted", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

#ifdef DEBUG
  // verify using known values when weighted is true
  float error;

  if (weighted)
    error = std::fabs(weight_j[0] - 0.306667) +
            std::fabs(weight_j[1] - 0.000000) +
            std::fabs(weight_j[2] - 3.680000) +
            std::fabs(weight_j[3] - 1.380000) +
            std::fabs(weight_j[4] - 0.788571) +
            std::fabs(weight_j[5] - 0.460000);

  else
    error = std::fabs(weight_j[0] - 0.230000) +
            std::fabs(weight_j[1] - 0.000000) +
            std::fabs(weight_j[2] - 3.680000) +
            std::fabs(weight_j[3] - 1.380000) +
            std::fabs(weight_j[4] - 0.920000) +
            std::fabs(weight_j[5] - 0.460000);

  if (error > 1e-5) {
    for (int i = 0; i < e; i++) printf("wj: %d %f\n", i, weight_j[i]);
    printf("FAIL");
  } else {
    printf("PASS");
  }
  printf("\n");
#endif

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  cudaFree (d_work);
  cudaFree (d_weight_i);
  cudaFree (d_weight_s);
  cudaFree (d_weight_j);
  cudaFree (d_csrInd);
  cudaFree (d_csrVal);
  cudaFree (d_csrPtr);
  free(weight_j);
#ifdef DEBUG
  free(weight_i);
  free(weight_s);
  free(work);
#endif
}

// Utilities
void printMatrix(const matrix& M)
{
  int m = M.size();
  int n = M[0].size();
  for (int i = 0; i < m; i++) {
    for (int j = 0; j < n; j++)
      cout << M[i][j] << " ";
    cout << endl;
  }
}

template <typename T>
void printVector(const vector<T>& V, char* msg)
{
  cout << msg << "[ ";
  for_each(V.begin(), V.end(), [](int a) { cout << a << " "; });
  cout << "]" << endl;
}

// Reference: https://www.geeksforgeeks.org/sparse-matrix-representations-set-3-csr/
int main(int argc, char** argv)
{
  int iteration = 10;

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate command-line configuration at program entry
  {
#ifdef DEBUG
    bool _t1_pass = true;
#else
    bool _t1_pass = (argc >= 4);
#endif
    printf("LASSI-VERIFY ASSERTIONS [main_args_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

#ifdef DEBUG
  matrix M  = {
    { 0, 0, 0, 1},
    { 5, 8, 0, 0},
    { 0, 0, 3, 0},
    { 0, 6, 0, 1}
  };
#else

  int numRow = atoi(argv[1]);
  int numCol = atoi(argv[2]);
  iteration = atoi(argv[3]);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate parsed matrix dimensions and iteration count
  {
    bool _t1_pass = (numRow > 0 && numCol > 0 && iteration > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_dims_iteration_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  srand(2);

  matrix M;
  vector<vtype> rowElems(numCol);
  for (int r = 0; r < numRow; r++) {
    for (int c = 0; c < numCol; c++)
      rowElems[c] = rand() % 10;
    M.push_back(rowElems);
  }
#endif

  int row = M.size();
  int col = M[0].size();

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate constructed matrix dimensions and row consistency
  {
    bool _t1_pass = (row > 0 && col > 0);
    if (_t1_pass) {
      for (int i = 0; i < row; i++) {
        if ((int)M[i].size() != col) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_matrix_shape_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate matrix entries are finite before CSR conversion
  {
    bool _t1_pass = true;
    for (int i = 0; i < row && _t1_pass; i++) {
      for (int j = 0; j < col; j++) {
        if (!std::isfinite((double)M[i][j])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_matrix_finite]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  printf("Number of matrix rows and cols: %d %d\n", row, col);
  vector<vtype> csr_val;
  vector<int> csr_ptr = { 0 }; // require -std=c++11
  vector<int> csr_ind;
  int nnz = 0; // count Number of non-zero elements in each row

  for (int i = 0; i < row; i++) {
    for (int j = 0; j < col; j++) {
      if (M[i][j] != (vtype)0) {
        csr_val.push_back(M[i][j]);
        csr_ind.push_back(j);
        nnz++;
      }
    }
    csr_ptr.push_back(nnz);
  }

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [csr_values_host] (do not edit or delete this block) >>>
  {
    // --- LASSI-VERIFY PASSTHROUGH: capture constructed CSR values buffer before GPU execution ---
    vtype* lv_host_buf = (vtype*) malloc(nnz * sizeof(vtype));
    for (int lv_i = 0; lv_i < nnz; lv_i++) lv_host_buf[lv_i] = csr_val[lv_i];

    vtype lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
    double lv_sum = 0.0;
    for (int lv_i = 0; lv_i < nnz; lv_i++) {
      if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
      if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
      lv_sum += (double)lv_host_buf[lv_i];
    }
    double lv_mean = lv_sum / nnz;

    printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [csr_values_host] ===\n");
    printf("Buffer: csr_val\n");
    printf("Type: float\n");
    printf("Elements: %d\n", (int)nnz);
    printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
           (double)lv_min, (double)lv_max, lv_mean);

    int lv_sample = (nnz < 10) ? (int)nnz : 10;
    printf("Sample_first10:");
    for (int lv_i = 0; lv_i < lv_sample; lv_i++)
      printf(" %.17g", (double)lv_host_buf[lv_i]);
    printf("\n");

    if (nnz > 10) {
      printf("Sample_last10:");
      for (int lv_i = (int)nnz - 10; lv_i < (int)nnz; lv_i++)
        printf(" %.17g", (double)lv_host_buf[lv_i]);
      printf("\n");
    }
    printf("=== END LASSI-VERIFY PASSTHROUGH [csr_values_host] ===\n");
    free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate CSR arrays after construction and before GPU use
  {
    bool _t1_pass = true;
    if ((int)csr_ptr.size() != row + 1) _t1_pass = false;
    if (_t1_pass && (int)csr_ind.size() != nnz) _t1_pass = false;
    if (_t1_pass && (int)csr_val.size() != nnz) _t1_pass = false;
    if (_t1_pass && csr_ptr[0] != 0) _t1_pass = false;
    if (_t1_pass && csr_ptr[row] != nnz) _t1_pass = false;
    for (int i = 0; i < row && _t1_pass; i++) {
      if (!(csr_ptr[i] <= csr_ptr[i+1])) { _t1_pass = false; break; }
    }
    for (int i = 0; i < nnz && _t1_pass; i++) {
      if (!(csr_ind[i] >= 0 && csr_ind[i] < col)) { _t1_pass = false; break; }
      if (!std::isfinite((double)csr_val[i])) { _t1_pass = false; break; }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_csr_valid_before_gpu]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // print when the matrix is small
  if (row <= 16 && col <= 16) {
    printMatrix(M);
    printVector(csr_val, (char*)"values = ");
    printVector(csr_ptr, (char*)"row pointer = ");
    printVector(csr_ind, (char*)"col indices = ");
  }

  jaccard_weight<true, vtype>(iteration, row, nnz, csr_ptr.data(), csr_ind.data(), csr_val.data());
  jaccard_weight<false, vtype>(iteration, row, nnz, csr_ptr.data(), csr_ind.data(), csr_val.data());

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Validate final host-side structural properties at program exit
  {
    bool _t1_pass = (row > 0 && col > 0 && nnz >= 0 && csr_ptr[row] == nnz);
    printf("LASSI-VERIFY ASSERTIONS [main_exit_structure_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  return 0;
}
```

... (157 outputs left out)

```cuda

/*
 * Copyright (c) 2019, NVIDIA CORPORATION.
 *
 * Licensed under the Apache License, Version 2.0 (the "License");
 * you may not use this file except in compliance with the License.
 * You may obtain a copy of the License at
 *
 *     http://www.apache.org/licenses/LICENSE-2.0
 *
 * Unless required by applicable law or agreed to in writing, software
 * distributed under the License is distributed on an "AS IS" BASIS,
 * WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
 * See the License for the specific language governing permissions and
 * limitations under the License.
 */

#include <stdio.h>
#include <cstdio>
#include <algorithm>
#include <cmath>
#include <iostream>
#include <vector>
#include <chrono>
#include <cstdlib>
#include <cuda.h>
#include <cuda_runtime.h>

using namespace std;

#define MAX_KERNEL_THREADS 256
#define WARP_SIZE 32
#define FULL_MASK 0xFFFFFFFFu

// float or double
typedef float vtype;
typedef vector<vector<vtype>> matrix;

#define CUDA_CHECK(call)                                                         \
  do {                                                                           \
    cudaError_t err__ = (call);                                                  \
    if (err__ != cudaSuccess) {                                                  \
      fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__,           \
              cudaGetErrorString(err__));                                        \
      exit(EXIT_FAILURE);                                                        \
    }                                                                            \
  } while (0)

template <typename T>
__device__ __forceinline__ T warp_reduce_sum(T val) {
  for (int offset = WARP_SIZE / 2; offset > 0; offset >>= 1) {
    val += __shfl_down_sync(FULL_MASK, val, offset);
  }
  return val;
}

template <typename T>
__global__ void row_sum_kernel_weighted(
    const int n,
    const int *__restrict__ csrPtr,
    const int *__restrict__ csrInd,
    const T *__restrict__ w,
    T *__restrict__ work)
{
  const int warps_per_block = blockDim.x / WARP_SIZE;
  const int warp_id_in_block = threadIdx.x / WARP_SIZE;
  const int lane = threadIdx.x & (WARP_SIZE - 1);
  const int global_warp_id = blockIdx.x * warps_per_block + warp_id_in_block;
  const int total_warps = gridDim.x * warps_per_block;

  for (int row = global_warp_id; row < n; row += total_warps) {
    const int start = csrPtr[row];
    const int end = csrPtr[row + 1];

    T sum = (T)0;
    for (int idx = start + lane; idx < end; idx += WARP_SIZE) {
      sum += w[csrInd[idx]];
    }
    sum = warp_reduce_sum(sum);

    if (lane == 0) work[row] = sum;
  }
}

template <typename T>
__global__ void row_sum_kernel_unweighted(
    const int n,
    const int *__restrict__ csrPtr,
    T *__restrict__ work)
{
  const int tid = blockIdx.x * blockDim.x + threadIdx.x;
  const int stride = gridDim.x * blockDim.x;

  for (int row = tid; row < n; row += stride) {
    work[row] = (T)(csrPtr[row + 1] - csrPtr[row]);
  }
}

template <bool weighted, typename T>
__global__ void fill_kernel(const int e, T* __restrict__ w, const T value)
{
  const int tid = blockIdx.x * blockDim.x + threadIdx.x;
  const int stride = gridDim.x * blockDim.x;

  for (int j = tid; j < e; j += stride) {
    w[j] = weighted ? (T)(j + 1) / (T)e : value;
  }
}

template <bool weighted, typename T>
__global__ void jaccard_is_edge_parallel(
    const int e,
    const int *__restrict__ csrPtr,
    const int *__restrict__ csrInd,
    const T *__restrict__ v,
    const T *__restrict__ work,
    T *__restrict__ weight_i,
    T *__restrict__ weight_s)
{
  const int warps_per_block = blockDim.x / WARP_SIZE;
  const int warp_id_in_block = threadIdx.x / WARP_SIZE;
  const int lane = threadIdx.x & (WARP_SIZE - 1);
  const int global_warp_id = blockIdx.x * warps_per_block + warp_id_in_block;
  const int total_warps = gridDim.x * warps_per_block;

  for (int edge = global_warp_id; edge < e; edge += total_warps) {
    int row = 0;
    int left = 0;
    int right = 0;

    left = 0;
    right = 0; // silence possible warnings for some compilers

    // Recover row from edge index via binary search over csrPtr.
    left = 0;
    right = 0;
    int lo = 0;
    int hi = 0;

    lo = 0;
    hi = 0;
    if (e > 0) {
      lo = 0;
      // number of rows is not explicitly passed, infer upper bound by searching valid ptr range
      // Here row lookup uses monotonic csrPtr and edge index. We need the row count upper bound.
      // Since csrPtr is not sized on device, use exponential/binary pattern over edge ownership:
      // we can directly binary search in a virtual range [0, e], but valid rows are bounded by csrPtr monotonic values.
      // A practical way is to search the smallest row s.t. csrPtr[row+1] > edge using edge as upper proxy.
      // However this still requires row count. To avoid that, we precompute edge->row separately would be costly.
      // Instead, launch a row-aware variant from host using edge row map.
    }
  }
}

template <bool weighted, typename T>
__global__ void jaccard_is_row_edge_parallel(
    const int n,
    const int *__restrict__ csrPtr,
    const int *__restrict__ csrInd,
    const T *__restrict__ v,
    const T *__restrict__ work,
    T *__restrict__ weight_i,
    T *__restrict__ weight_s)
{
  const int warps_per_block = blockDim.x / WARP_SIZE;
  const int warp_id_in_block = threadIdx.x / WARP_SIZE;
  const int lane = threadIdx.x & (WARP_SIZE - 1);
  const int global_warp_id = blockIdx.x * warps_per_block + warp_id_in_block;
  const int total_warps = gridDim.x * warps_per_block;

  for (int row = global_warp_id; row < n; row += total_warps) {
    const int row_start = csrPtr[row];
    const int row_end = csrPtr[row + 1];
    const int Ni = row_end - row_start;

    for (int edge = row_start; edge < row_end; ++edge) {
      const int col = csrInd[edge];
      const int col_start = csrPtr[col];
      const int col_end = csrPtr[col + 1];
      const int Nj = col_end - col_start;

      weight_s[edge] = work[row] + work[col];

      const int ref = (Ni < Nj) ? row : col;
      const int cur = (Ni < Nj) ? col : row;

      const int ref_start = csrPtr[ref];
      const int ref_end = csrPtr[ref + 1];
      const int cur_start = csrPtr[cur];
      const int cur_end = csrPtr[cur + 1];

      T local_sum = (T)0;

      for (int i = ref_start + lane; i < ref_end; i += WARP_SIZE) {
        const int ref_col = csrInd[i];
        const T ref_val = weighted ? v[ref_col] : (T)1.0;

        int l = cur_start;
        int r = cur_end - 1;
        while (l <= r) {
          const int m = (l + r) >> 1;
          const int cur_col = csrInd[m];
          if (cur_col < ref_col) {
            l = m + 1;
          } else if (cur_col > ref_col) {
            r = m - 1;
          } else {
            local_sum += ref_val;
            break;
          }
        }
      }

      local_sum = warp_reduce_sum(local_sum);
      if (lane == 0) {
        weight_i[edge] = local_sum;
      }
    }
  }
}

template <bool weighted, typename T>
__global__ void jaccard_jw(
    const int e,
    const T *__restrict__ csrVal,
    const T gamma,
    const T *__restrict__ weight_i,
    const T *__restrict__ weight_s,
    T *__restrict__ weight_j)
{
  const int tid = blockIdx.x * blockDim.x + threadIdx.x;
  const int stride = gridDim.x * blockDim.x;

  for (int j = tid; j < e; j += stride) {
    T Wi = weight_i[j];
    T Ws = weight_s[j];
    weight_j[j] = (gamma * csrVal[j]) * (Wi / (Ws - Wi));
  }
}

template <bool weighted, typename T>
void jaccard_weight(const int iteration, const int n, const int e,
                    int* csr_ptr, int* csr_ind, T* csr_val)
{
  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Function entry pointer and dimension validity for jaccard_weight
  {
    bool _t1_pass = (csr_ptr != NULL && csr_ind != NULL && csr_val != NULL &&
                     iteration > 0 && n >= 0 && e >= 0);
    printf("LASSI-VERIFY ASSERTIONS [jw_entry_params_valid_%s]: %s\n",
           weighted ? "weighted" : "unweighted",
           _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - CSR row pointer structural validity at function entry
  {
    bool _t1_pass = true;
    if (csr_ptr == NULL || n < 0 || e < 0) {
      _t1_pass = false;
    } else {
      if (csr_ptr[0] != 0 || csr_ptr[n] != e) _t1_pass = false;
      for (int i = 0; _t1_pass && i < n; i++) {
        if (csr_ptr[i] > csr_ptr[i+1]) _t1_pass = false;
        if (csr_ptr[i] < 0 || csr_ptr[i+1] < 0) _t1_pass = false;
        if (csr_ptr[i] > e || csr_ptr[i+1] > e) _t1_pass = false;
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_csr_ptr_valid_%s]: %s\n",
           weighted ? "weighted" : "unweighted",
           _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - CSR column indices and values validity before host-to-device transfers
  {
    bool _t1_pass = true;
    if (csr_ind == NULL || csr_val == NULL || n < 0 || e < 0) {
      _t1_pass = false;
    } else {
      for (int i = 0; i < e; i++) {
        if (csr_ind[i] < 0 || csr_ind[i] >= n) { _t1_pass = false; break; }
        if (!std::isfinite((double)csr_val[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_input_arrays_valid_%s]: %s\n",
           weighted ? "weighted" : "unweighted",
           _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  const T gamma = (T)0.46;  // arbitrary

  T *d_weight_i, *d_weight_s, *d_weight_j, *d_work;
  int *d_csrInd, *d_csrPtr;
  T *d_csrVal;

#ifdef DEBUG
  T* weight_i = (T*)malloc(sizeof(T) * e);
  T* weight_s = (T*)malloc(sizeof(T) * e);
  T* work = (T*)malloc(sizeof(T) * n);
#endif
  T* weight_j = (T*)malloc(sizeof(T) * e);

  CUDA_CHECK(cudaMalloc((void**)&d_work, sizeof(T) * n));
  CUDA_CHECK(cudaMalloc((void**)&d_weight_i, sizeof(T) * e));
  CUDA_CHECK(cudaMalloc((void**)&d_weight_s, sizeof(T) * e));
  CUDA_CHECK(cudaMalloc((void**)&d_weight_j, sizeof(T) * e));
  CUDA_CHECK(cudaMalloc((void**)&d_csrVal, sizeof(T) * e));
  CUDA_CHECK(cudaMalloc((void**)&d_csrPtr, sizeof(int) * (n + 1)));
  CUDA_CHECK(cudaMalloc((void**)&d_csrInd, sizeof(int) * e));

  CUDA_CHECK(cudaMemcpy(d_csrPtr, csr_ptr, sizeof(int) * (n + 1), cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_csrInd, csr_ind, sizeof(int) * e, cudaMemcpyHostToDevice));
  CUDA_CHECK(cudaMemcpy(d_csrVal, csr_val, sizeof(T) * e, cudaMemcpyHostToDevice));

  cudaStream_t stream;
  CUDA_CHECK(cudaStreamCreate(&stream));

  CUDA_CHECK(cudaDeviceSynchronize());
  auto start = std::chrono::steady_clock::now();

  const int block_fill = MAX_KERNEL_THREADS;
  const int grid_fill = max(1, (e + block_fill - 1) / block_fill);

  const int block_row_sum_weighted = 128;
  const int warps_per_block_row_sum = block_row_sum_weighted / WARP_SIZE;
  const int grid_row_sum_weighted = max(1, (n + warps_per_block_row_sum - 1) / warps_per_block_row_sum);

  const int block_row_sum_unweighted = MAX_KERNEL_THREADS;
  const int grid_row_sum_unweighted = max(1, (n + block_row_sum_unweighted - 1) / block_row_sum_unweighted);

  const int block_is = 128;
  const int warps_per_block_is = block_is / WARP_SIZE;
  const int grid_is = max(1, (n + warps_per_block_is - 1) / warps_per_block_is);

  const int block_jw = MAX_KERNEL_THREADS;
  const int grid_jw = max(1, (e + block_jw - 1) / block_jw);

  for (int i = 0; i < iteration; i++) {
    fill_kernel<weighted, T><<<grid_fill, block_fill, 0, stream>>>(e, d_weight_j, (T)1.0);
    fill_kernel<false, T><<<grid_fill, block_fill, 0, stream>>>(e, d_weight_i, (T)0.0);

    if (weighted) {
      row_sum_kernel_weighted<T><<<grid_row_sum_weighted, block_row_sum_weighted, 0, stream>>>(
          n, d_csrPtr, d_csrInd, d_weight_j, d_work);
    } else {
      row_sum_kernel_unweighted<T><<<grid_row_sum_unweighted, block_row_sum_unweighted, 0, stream>>>(
          n, d_csrPtr, d_work);
    }

    jaccard_is_row_edge_parallel<weighted, T><<<grid_is, block_is, 0, stream>>>(
        n, d_csrPtr, d_csrInd, d_weight_j, d_work, d_weight_i, d_weight_s);

    jaccard_jw<weighted, T><<<grid_jw, block_jw, 0, stream>>>(
        e, d_csrVal, gamma, d_weight_i, d_weight_s, d_weight_j);
  }

  CUDA_CHECK(cudaStreamSynchronize(stream));
  auto end = std::chrono::steady_clock::now();
  auto time = std::chrono::duration_cast<std::chrono::nanoseconds>(end - start).count();
  cout << "Average execution time of kernels: " << (time * 1e-9f) / iteration << " (s)\n";

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [final_device_weight_j] (do not edit or delete this block) >>>
  {
    // --- LASSI-VERIFY PASSTHROUGH: capture final jaccard weight output buffer from device ---
    float* lv_host_buf = (float*) malloc(e * sizeof(float));
    cudaMemcpy(lv_host_buf, d_weight_j, e * sizeof(float), cudaMemcpyDeviceToHost);

    float lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
    double lv_sum = 0.0;
    for (int lv_i = 0; lv_i < e; lv_i++) {
      if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
      if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
      lv_sum += (double)lv_host_buf[lv_i];
    }
    double lv_mean = lv_sum / e;

    printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [final_device_weight_j] ===\n");
    printf("Buffer: d_weight_j\n");
    printf("Type: float\n");
    printf("Elements: %d\n", (int)e);
    printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
           (double)lv_min, (double)lv_max, lv_mean);

    int lv_sample = (e < 10) ? (int)e : 10;
    printf("Sample_first10:");
    for (int lv_i = 0; lv_i < lv_sample; lv_i++)
      printf(" %.17g", (double)lv_host_buf[lv_i]);
    printf("\n");

    if (e > 10) {
      printf("Sample_last10:");
      for (int lv_i = (int)e - 10; lv_i < (int)e; lv_i++)
        printf(" %.17g", (double)lv_host_buf[lv_i]);
      printf("\n");
    }
    printf("=== END LASSI-VERIFY PASSTHROUGH [final_device_weight_j] ===\n");
    free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  CUDA_CHECK(cudaMemcpy(weight_j, d_weight_j, sizeof(T) * e, cudaMemcpyDeviceToHost));

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [final_host_weight_j] (do not edit or delete this block) >>>
  {
    // --- LASSI-VERIFY PASSTHROUGH: capture final host-visible jaccard weight output buffer ---
    float* lv_host_buf = (float*) malloc(e * sizeof(float));
    cudaMemcpy(lv_host_buf, d_weight_j, e * sizeof(float), cudaMemcpyDeviceToHost);

    float lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
    double lv_sum = 0.0;
    for (int lv_i = 0; lv_i < e; lv_i++) {
      if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
      if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
      lv_sum += (double)lv_host_buf[lv_i];
    }
    double lv_mean = lv_sum / e;

    printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [final_host_weight_j] ===\n");
    printf("Buffer: d_weight_j\n");
    printf("Type: float\n");
    printf("Elements: %d\n", (int)e);
    printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
           (double)lv_min, (double)lv_max, lv_mean);

    int lv_sample = (e < 10) ? (int)e : 10;
    printf("Sample_first10:");
    for (int lv_i = 0; lv_i < lv_sample; lv_i++)
      printf(" %.17g", (double)lv_host_buf[lv_i]);
    printf("\n");

    if (e > 10) {
      printf("Sample_last10:");
      for (int lv_i = (int)e - 10; lv_i < (int)e; lv_i++)
        printf(" %.17g", (double)lv_host_buf[lv_i]);
      printf("\n");
    }
    printf("=== END LASSI-VERIFY PASSTHROUGH [final_host_weight_j] ===\n");
    free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Output numerical validity after device-to-host transfer
  {
    bool _t1_pass = (weight_j != NULL);
    if (_t1_pass) {
      for (int i = 0; i < e; i++) {
        if (!std::isfinite((double)weight_j[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_output_finite_%s]: %s\n",
           weighted ? "weighted" : "unweighted",
           _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

#ifdef DEBUG
  CUDA_CHECK(cudaMemcpy(weight_i, d_weight_i, sizeof(T) * e, cudaMemcpyDeviceToHost));
  CUDA_CHECK(cudaMemcpy(weight_s, d_weight_s, sizeof(T) * e, cudaMemcpyDeviceToHost));
  CUDA_CHECK(cudaMemcpy(work, d_work, sizeof(T) * n, cudaMemcpyDeviceToHost));

  float error;

  if (weighted)
    error = std::fabs(weight_j[0] - 0.306667f) +
            std::fabs(weight_j[1] - 0.000000f) +
            std::fabs(weight_j[2] - 3.680000f) +
            std::fabs(weight_j[3] - 1.380000f) +
            std::fabs(weight_j[4] - 0.788571f) +
            std::fabs(weight_j[5] - 0.460000f);
  else
    error = std::fabs(weight_j[0] - 0.230000f) +
            std::fabs(weight_j[1] - 0.000000f) +
            std::fabs(weight_j[2] - 3.680000f) +
            std::fabs(weight_j[3] - 1.380000f) +
            std::fabs(weight_j[4] - 0.920000f) +
            std::fabs(weight_j[5] - 0.460000f);

  if (error > 1e-5) {
    for (int i = 0; i < e; i++) printf("wj: %d %f\n", i, weight_j[i]);
    printf("FAIL");
  } else {
    printf("PASS");
  }
  printf("\n");
#endif

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Function exit validity for final host-visible result buffer
  {
    bool _t1_pass = (weight_j != NULL && e >= 0);
    if (_t1_pass) {
      for (int i = 0; i < e; i++) {
        if (!std::isfinite((double)weight_j[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [jw_exit_result_valid_%s]: %s\n",
           weighted ? "weighted" : "unweighted",
           _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete_jaccard_weight_%s]: PASS\n",
           weighted ? "weighted" : "unweighted"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  CUDA_CHECK(cudaFree(d_work));
  CUDA_CHECK(cudaFree(d_weight_i));
  CUDA_CHECK(cudaFree(d_weight_s));
  CUDA_CHECK(cudaFree(d_weight_j));
  CUDA_CHECK(cudaFree(d_csrInd));
  CUDA_CHECK(cudaFree(d_csrVal));
  CUDA_CHECK(cudaFree(d_csrPtr));
  CUDA_CHECK(cudaStreamDestroy(stream));

  free(weight_j);
#ifdef DEBUG
  free(weight_i);
  free(weight_s);
  free(work);
#endif
}

// Utilities
void printMatrix(const matrix& M)
{
  int m = (int)M.size();
  int n = (int)M[0].size();
  for (int i = 0; i < m; i++) {
    for (int j = 0; j < n; j++)
      cout << M[i][j] << " ";
    cout << endl;
  }
}

template <typename T>
void printVector(const vector<T>& V, char* msg)
{
  cout << msg << "[ ";
  for_each(V.begin(), V.end(), [](T a) { cout << a << " "; });
  cout << "]" << endl;
}

// Reference: https://www.geeksforgeeks.org/sparse-matrix-representations-set-3-csr/
int main(int argc, char** argv)
{
  int iteration = 10;

#ifdef DEBUG
  matrix M = {
    { 0, 0, 0, 1},
    { 5, 8, 0, 0},
    { 0, 0, 3, 0},
    { 0, 6, 0, 1}
  };
#else
  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Command line argument count validity at main entry
  {
    bool _t1_pass = (argc >= 4);
    printf("LASSI-VERIFY ASSERTIONS [main_argc_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  if (argc < 4) {
    fprintf(stderr, "Usage: %s <numRow> <numCol> <iteration>\n", argv[0]);
    return EXIT_FAILURE;
  }

  int numRow = atoi(argv[1]);
  int numCol = atoi(argv[2]);
  iteration = atoi(argv[3]);

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Input dimensions and iteration count validity after parsing
  {
    bool _t1_pass = (numRow > 0 && numCol > 0 && iteration > 0);
    printf("LASSI-VERIFY ASSERTIONS [main_dims_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  srand(2);

  matrix M;
  vector<vtype> rowElems(numCol);
  for (int r = 0; r < numRow; r++) {
    for (int c = 0; c < numCol; c++)
      rowElems[c] = (vtype)(rand() % 10);
    M.push_back(rowElems);
  }
#endif

  int row = (int)M.size();
  int col = (int)M[0].size();
  printf("Number of matrix rows and cols: %d %d\n", row, col);

  vector<vtype> csr_val;
  vector<int> csr_ptr = {0};
  vector<int> csr_ind;
  int nnz = 0;

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Dense matrix shape validity before CSR construction
  {
    bool _t1_pass = (row > 0 && col > 0);
    if (_t1_pass) {
      for (int i = 0; i < row; i++) {
        if ((int)M[i].size() != col) { _t1_pass = false; break; }
        for (int j = 0; j < col; j++) {
          if (!std::isfinite((double)M[i][j])) { _t1_pass = false; break; }
        }
        if (!_t1_pass) break;
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_matrix_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  for (int i = 0; i < row; i++) {
    for (int j = 0; j < col; j++) {
      if (M[i][j] != (vtype)0) {
        csr_val.push_back(M[i][j]);
        csr_ind.push_back(j);
        nnz++;
      }
    }
    csr_ptr.push_back(nnz);
  }

  // >>> BEGIN LASSI-VERIFY PASSTHROUGH [csr_values_host] (do not edit or delete this block) >>>
  {
    // --- LASSI-VERIFY PASSTHROUGH: capture CSR values buffer after CSR construction ---
    float* lv_host_buf = (float*) malloc(nnz * sizeof(float));
    for (int lv_i = 0; lv_i < nnz; lv_i++) lv_host_buf[lv_i] = csr_val[lv_i];

    float lv_min = lv_host_buf[0], lv_max = lv_host_buf[0];
    double lv_sum = 0.0;
    for (int lv_i = 0; lv_i < nnz; lv_i++) {
      if (lv_host_buf[lv_i] < lv_min) lv_min = lv_host_buf[lv_i];
      if (lv_host_buf[lv_i] > lv_max) lv_max = lv_host_buf[lv_i];
      lv_sum += (double)lv_host_buf[lv_i];
    }
    double lv_mean = lv_sum / nnz;

    printf("\n=== BEGIN LASSI-VERIFY PASSTHROUGH [csr_values_host] ===\n");
    printf("Buffer: csr_val\n");
    printf("Type: float\n");
    printf("Elements: %d\n", (int)nnz);
    printf("Statistics: min=%.17g, max=%.17g, mean=%.17g\n",
           (double)lv_min, (double)lv_max, lv_mean);

    int lv_sample = (nnz < 10) ? (int)nnz : 10;
    printf("Sample_first10:");
    for (int lv_i = 0; lv_i < lv_sample; lv_i++)
      printf(" %.17g", (double)lv_host_buf[lv_i]);
    printf("\n");

    if (nnz > 10) {
      printf("Sample_last10:");
      for (int lv_i = (int)nnz - 10; lv_i < (int)nnz; lv_i++)
        printf(" %.17g", (double)lv_host_buf[lv_i]);
      printf("\n");
    }
    printf("=== END LASSI-VERIFY PASSTHROUGH [csr_values_host] ===\n");
    free(lv_host_buf);
  }
  // <<< END LASSI-VERIFY PASSTHROUGH <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - CSR representation validity after construction and before GPU use
  {
    bool _t1_pass = ((int)csr_ptr.size() == row + 1 &&
                     (int)csr_ind.size() == nnz &&
                     (int)csr_val.size() == nnz &&
                     !csr_ptr.empty() &&
                     csr_ptr[0] == 0 &&
                     csr_ptr[row] == nnz);
    if (_t1_pass) {
      for (int i = 0; i < row; i++) {
        if (csr_ptr[i] > csr_ptr[i+1]) { _t1_pass = false; break; }
      }
      for (int i = 0; _t1_pass && i < nnz; i++) {
        if (csr_ind[i] < 0 || csr_ind[i] >= col) { _t1_pass = false; break; }
        if (!std::isfinite((double)csr_val[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_csr_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  if (row <= 16 && col <= 16) {
    printMatrix(M);
    printVector(csr_val, (char*)"values = ");
    printVector(csr_ptr, (char*)"row pointer = ");
    printVector(csr_ind, (char*)"col indices = ");
  }

  jaccard_weight<true, vtype>(iteration, row, nnz, csr_ptr.data(), csr_ind.data(), csr_val.data());
  jaccard_weight<false, vtype>(iteration, row, nnz, csr_ptr.data(), csr_ind.data(), csr_val.data());

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - Final host-side CSR buffers remain valid at main exit
  {
    bool _t1_pass = ((int)csr_ptr.size() == row + 1 &&
                     (int)csr_ind.size() == nnz &&
                     (int)csr_val.size() == nnz);
    if (_t1_pass) {
      for (int i = 0; i < row; i++) {
        if (csr_ptr[i] > csr_ptr[i+1]) { _t1_pass = false; break; }
      }
      for (int i = 0; _t1_pass && i < nnz; i++) {
        if (csr_ind[i] < 0 || csr_ind[i] >= col) { _t1_pass = false; break; }
        if (!std::isfinite((double)csr_val[i])) { _t1_pass = false; break; }
      }
    }
    printf("LASSI-VERIFY ASSERTIONS [main_exit_csr_valid]: %s\n", _t1_pass ? "PASS" : "FAIL"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  // >>> BEGIN LASSI-VERIFY ASSERTIONS (do not edit or delete this block) >>>
  // LASSI_INSTRUMENTATION - tier1_complete (sentinel — confirms all assertion blocks were reached)
  {
    printf("LASSI-VERIFY ASSERTIONS [tier1_complete_main]: PASS\n"); fflush(stdout);
  }
  // <<< END LASSI-VERIFY ASSERTIONS <<<

  return 0;
}

```

[Base LLM Verification] Compiling instrumented base LLM code ...
Compile command: ['nvcc', '-std=c++14', '-Xcompiler', '-Wall', '-arch=sm_89', '-O3', '/home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA__03-17-2026_t09_55_08/jaccard-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA_03-17-2026_t09_55_08_base_llm_verify.cu', '-o', '/home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA__03-17-2026_t09_55_08/jaccard-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA_03-17-2026_t09_55_08_base_llm_verify']

Compilation of code successful!

[Base LLM Verification] Executing ...
🔍 Using NVIDIA GPU for execution and power monitoring


Waiting for the system to settle down for 3 seconds...: 100%|██████████| 3/3 [00:03<00:00,  1.00s/it]


executable_file:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA__03-17-2026_t09_55_08/jaccard-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA_03-17-2026_t09_55_08_base_llm_verify
initial_dir:  /home/mdearing/Research/LASSI/LASSI-Verify
directory_path:  /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA__03-17-2026_t09_55_08
executable:  jaccard-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA_03-17-2026_t09_55_08_base_llm_verify
Working directory: /home/mdearing/Research/LASSI/LASSI-Verify/generated_codes/experiment/TEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA__03-17-2026_t09_55_08
Execute command: ['./jaccard-cuda_main_ee_refactorcode_CUDA_expTEST_FULLEXP_verify_0314-2026_gpt54_jaccard_runtm_NVIDIA_03-17-2026_t09_55_08_base_llm_verify'